# Arabic document VLM, part 2 (v6): the roadmap, phase by phase

**Runtime: GPU, L4** (*Runtime > Change runtime type > L4*). **Before this notebook:** run part 1b v6, which builds
`MyDrive/arabic_vlm/data_v6` (the frozen test suite, the filtered training sources and the forms). This notebook
starts from the v5 checkpoints (`stage3_v5`, `stage4_v5`) already on Drive; it does not repeat stages 1 and 2.

**How this notebook is organised.** Sections A–H are tools: run them at the start of every session (about 10–15
minutes, most of it copying the data folder to the Colab disk). Then run the phase you are on. Every experiment is
one call to `run_experiment(...)`: it trains, saves a checkpoint to Drive, scores it on the frozen test suite and
records everything in `results_v6.json`. Experiments that are already done are skipped, and an interrupted one
resumes from its last progress check, so a phase can run over several Colab sessions.

**Try a phase small first.** Set `RUN_SIZE = "pilot"` in section A to run a phase in minutes per experiment (a few
hundred training examples, 20 images per test set): it shows that every step works on your data before you spend
GPU hours. Pilot runs save under their own names (`v6_pilot_...`, `results_v6_pilot.json`), so switching back to
`"full"` trains everything properly.

| Phase | What runs | L4 hours (full size) | Decides |
| --- | --- | --- | --- |
| 0 | score v5 stage 3 and 4, Qwen2.5-VL-3B and Qwen3-VL-2B on the suite; optional label audit with AIN-7B | 4–5 | the baseline for every later number |
| 1 | decoding re-score, stage 3/stage 4 weight merge, schema-forced JSON, CLAHE | 3–4 | default decoding per task, the best merge, CLAHE yes or no |
| 2 | writer-diversity test (needs official KHATT), rendered handwriting lines, real handwriting with LoRA and with full fine-tuning, a short mixed stage | 70–90 | whether data or method moves unseen handwriting |
| 3 | Muharaf, then AraMS-28k and the memoirs, then a short mix | 20–30 | historical handwriting |
| 4 | English share raised, pages, GNHK | about 10 | English |
| 5 | forms as a separate adapter, scored with schema-forced decoding and by oracle crops | 6–10 | the forms recipe |
| 6 | checkpoint merges (early + final snapshot); an experimental Qwen3-VL-2B arm | a few | the recipe for the next full run |

**One change per run.** Each `run_experiment` changes one thing against its base (a data source, the method, the
decoding), so a gain can be credited. **The suite never changes.** Checks during training use validation splits
only, so picking the best check never touches a test set.

**What this notebook does not do yet** (listed in phase 6 with the reason): train a CRNN from scratch, GRPO
reinforcement learning, swap the decoder for Qwen3-0.6B, and two-step page reading with a real line detector.

**Nothing in this notebook has produced results yet.** Every number in the markdown is either from the v5 run or a
target from the roadmap.

## A. Setup and settings
Installs four small libraries, mounts Drive, sets every switch the notebook uses and copies the prepared data to
the Colab disk. The switches you may want to change are in the second cell: `RUN_SIZE` (`"pilot"`, `"short"` or
`"full"`), `SCORE_QWEN3`, and `FREEZE_VISION_BLOCKS` if full fine-tuning runs out of memory in phase 2.

In [ ]:
%pip install -q arabic-reshaper python-bidi bitsandbytes rapidfuzz

In [ ]:
import os, sys, re, gc, io, json, math, time, glob, random, shutil, hashlib, subprocess, importlib, unicodedata, zipfile
from collections import Counter, defaultdict
import numpy as np
import torch
from transformers import AutoProcessor, AutoImageProcessor, AutoConfig, Qwen2_5_VLForConditionalGeneration

# Colab ships an old torchao that makes PEFT (the LoRA library) fail to import. This notebook does not use torchao.
try:
    from peft.import_utils import is_torchao_available
    is_torchao_available()
except ImportError as e:
    if "torchao" in str(e):
        print("Removing an incompatible optional package:", e)
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
        importlib.invalidate_caches()
from peft import LoraConfig, get_peft_model, PeftModel

QUICK_TEST = os.environ.get("ARVL_QUICK_TEST", "0") == "1"   # tiny CPU run, used only to test this notebook

# --- where things are -----------------------------------------------------------------------
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    try:
        from google.colab import userdata
        os.environ.setdefault("HF_TOKEN", userdata.get("HF_TOKEN"))
    except Exception:
        pass
BASE_DIR = os.environ.get("ARVL_BASE", "/content/drive/MyDrive/arabic_vlm" if IN_COLAB else "arabic_vlm")
V5_STAGE3 = os.environ.get("ARVL_V5_STAGE3", os.path.join(BASE_DIR, "stage3_v5"))
V5_STAGE4 = os.environ.get("ARVL_V5_STAGE4", os.path.join(BASE_DIR, "stage4_v5"))
BASELINE_ID = os.environ.get("ARVL_BASELINE", "Qwen/Qwen2.5-VL-3B-Instruct")     # the model v5 compared against
QWEN3_VL_ID = os.environ.get("ARVL_QWEN3VL", "Qwen/Qwen3-VL-2B-Instruct")        # Apache 2.0; the candidate new base
AIN_ID = os.environ.get("ARVL_AIN", "MBZUAI/AIN")                                 # MIT; Arabic VLM used as a second reader

# --- run size and switches ---------------------------------------------------------------------
RUN_SIZE = "full"                       # "pilot" (minutes, to check a phase runs end to end), "short" (an hour or two), "full"
SCORE_QWEN3 = True                      # phase 0: also score Qwen3-VL-2B zero-shot (needs transformers >= 4.57)
MIN_TOKENS, MAX_TOKENS, MAX_TOKENS_INFER = 32, 1024, 1280   # visual tokens per image in training/evaluation, and the larger inference limit
LORA_R, LORA_ALPHA = 32, 64
VISION_FULL_BLOCKS = 8                  # LoRA runs: the last 8 vision blocks train in full (the v5 recipe)
LR_FULL_BLOCKS = 2e-5                   # ... at this learning rate
FULL_LR = {"decoder": 1e-4, "projector": 1e-4, "vision": 1e-5}   # full fine-tuning: per-part learning rates (2026 winner: 1e-4 / 9e-6)
FREEZE_VISION_BLOCKS = 0                # full fine-tuning: freeze the first N vision blocks if memory runs out (the roadmap's fallback: 8)
OPTIMIZER_8BIT = True                   # full fine-tuning: bitsandbytes paged 8-bit AdamW (about 2.3 GB of optimizer state instead of 9.3)
SAVE_BF16 = True                        # checkpoints in bf16: 2.3 GB each instead of 4.6 (training still runs on fp32 weights)
CHECKS_PER_RUN = 10                     # progress checks per experiment (validation sets only)
N_CHECK = 24                            # images per validation set at a check
SUITE_N = None                          # cap images per suite set below what part 1b chose (None = as frozen, up to 500)
VD_MAX = 200                            # images per set for the visual-dependence (swapped-image) test
OUTPUT_CHAR_MAP = str.maketrans({"ی": "ي", "ک": "ك"})   # Farsi yeh and kaf -> Arabic, applied to every model's output
if RUN_SIZE == "pilot":                 # a pilot checks that a phase runs, not what it scores: few checks, 20 images per test set
    N_CHECK, CHECKS_PER_RUN, VD_MAX, SUITE_N = 8, 2, 20, 20
if QUICK_TEST:
    N_CHECK, CHECKS_PER_RUN, VD_MAX, SUITE_N = 2, 2, 4, 3
    VISION_FULL_BLOCKS = 1
# Pilot and short runs keep their own checkpoints, predictions and results (v6_pilot_..., results_v6_pilot.json), so a
# full run never finds a pilot's checkpoint under its own name and skips training.
RUN_NAME = "v6" if RUN_SIZE == "full" else f"v6_{RUN_SIZE}"
RESULTS_PATH = os.path.join(BASE_DIR, f"results_{RUN_NAME}.json")
PREDICTIONS_DIR = os.path.join(BASE_DIR, f"{RUN_NAME}_predictions")
os.makedirs(PREDICTIONS_DIR, exist_ok=True)

# --- hardware ---------------------------------------------------------------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 2**30 if DEVICE.type == "cuda" else 0
USE_AMP = DEVICE.type == "cuda"
AMP_DTYPE = torch.bfloat16 if (USE_AMP and torch.cuda.is_bf16_supported(including_emulation=False)) else (torch.float16 if USE_AMP else torch.float32)
print(f"run {RUN_NAME}: checkpoints and results in {BASE_DIR} | GPU {GPU_GB:.0f} GB | mixed precision {AMP_DTYPE if USE_AMP else 'off'}")
for label, path in [("v5 stage 3", V5_STAGE3), ("v5 stage 4", V5_STAGE4)]:
    print(f"   {label}: {'found' if os.path.isdir(path) else 'MISSING'} ({path})")

# --- the prepared data (part 1b v6), copied to the Colab disk ---------------------------------------
DATA_ON_DRIVE = os.environ.get("ARVL_DATA", os.path.join(BASE_DIR, "data_v6"))
DATA_DIR = "/content/arabic_vlm_data_v6" if IN_COLAB else os.environ.get("ARVL_DATA_LOCAL", DATA_ON_DRIVE)
if not os.path.exists(os.path.join(DATA_DIR, "manifest.json")):
    start = time.time()
    assert os.path.isdir(DATA_ON_DRIVE), f"No prepared data at {DATA_ON_DRIVE}: run part 1b v6 first."
    shutil.copytree(DATA_ON_DRIVE, DATA_DIR, ignore=shutil.ignore_patterns("raw", "forms"), dirs_exist_ok=True)
    print(f"data copied to {DATA_DIR} in {(time.time() - start) / 60:.1f} min (raw downloads and printouts left on Drive)")
MANIFEST = json.load(open(os.path.join(DATA_DIR, "manifest.json")))
assert MANIFEST.get("version") == "v6", f"this notebook needs the v6 data folder from part 1b v6, found {MANIFEST.get('version')}"
SUITE = json.load(open(os.path.join(DATA_DIR, "eval_suite.json")))
print(f"data: version {MANIFEST['version']}, updated {MANIFEST.get('updated')}; suite with {len(SUITE['sets'])} sets")


def results_file():
    return RESULTS_PATH, (json.load(open(RESULTS_PATH)) if os.path.exists(RESULTS_PATH) else {})


def save_results(**updates):
    path, results = results_file()
    results.update(updates)
    tmp = path + ".tmp"
    json.dump(results, open(tmp, "w"), indent=1, ensure_ascii=False)
    os.replace(tmp, path)
    return path


# Decisions made in phase 1 (decoding per task, CLAHE, schema JSON) are saved and reused in every later session.
DEFAULTS = {"line_decoding": {}, "form_schema": False, "clahe_sets": [], **results_file()[1].get("defaults", {})}
print("defaults:", DEFAULTS)

## B. Drawing text
As in v5: `render_lines` draws lines in four styles (page, colour, slide, sign) and `damage` makes an image look
scanned or photographed. Arabic is laid out by Pillow's raqm engine (joined letters, right to left, diacritics in
place). **New in v6:**

- The six fonts part 1b holds out for the form test are removed from every list here, so no training image uses them.
- `render_handwriting(text)` draws a line in a handwriting-like font and then makes it look written: a random slant,
  a wavy baseline, elastic wobble of the strokes, a thicker or thinner pen, uneven ink pressure, a pen colour, and
  paper with grain, tint and sometimes ruled lines. Phase 2 pre-trains on hundreds of thousands of these.

In [ ]:
import cv2
from PIL import Image, ImageDraw, ImageFont, ImageFilter, ImageOps, features

cv2.setNumThreads(1)                     # the data loader workers each run their own copy
FONT_DIR = os.path.join(DATA_DIR, "fonts")
FONT_LIST = json.load(open(os.path.join(FONT_DIR, "fonts.json")))
HELD_OUT_FONTS = set(FONT_LIST.get("heldout_forms", []))
font_path = lambda name: os.path.join(FONT_DIR, name)
usable = lambda names: [font_path(n) for n in names if n not in HELD_OUT_FONTS]
SPECIAL = set(FONT_LIST["handwriting_like"]) | set(FONT_LIST["display"]) | set(FONT_LIST.get("mono", []))
ARABIC_BODY_FONTS = usable(f for f in FONT_LIST["arabic"] if f not in SPECIAL)
ARABIC_SPECIAL_FONTS = usable(f for f in FONT_LIST["arabic"] if f in SPECIAL)
ENGLISH_BODY_FONTS = usable(f for f in FONT_LIST["english"] if f not in SPECIAL)
ENGLISH_SPECIAL_FONTS = usable(f for f in FONT_LIST["english"] if f in SPECIAL)
DISPLAY_FONTS = {"ar": usable(f for f in FONT_LIST["display"] if f in FONT_LIST["arabic"]) or ARABIC_BODY_FONTS,
                 "en": usable(f for f in FONT_LIST["display"] if f in FONT_LIST["english"]) or ENGLISH_BODY_FONTS}
MONO_FONTS = usable(FONT_LIST.get("mono", [])) or ENGLISH_BODY_FONTS
HAND_FONTS = {"ar": usable(f for f in FONT_LIST["handwriting_like"] if f in FONT_LIST["arabic"]) or ARABIC_BODY_FONTS,
              "en": usable(f for f in FONT_LIST["handwriting_like"] if f in FONT_LIST["english"]) or ENGLISH_BODY_FONTS}
HAS_RAQM = features.check("raqm")
print(f"fonts for training: {len(ARABIC_BODY_FONTS)} Arabic body, {len(HAND_FONTS['ar'])} Arabic handwriting-like, "
      f"{len(ENGLISH_BODY_FONTS)} English body, {len(HAND_FONTS['en'])} English handwriting-like, {len(MONO_FONTS)} mono "
      f"(held out for the form test: {', '.join(sorted(HELD_OUT_FONTS))}) | raqm: {HAS_RAQM}")
_fonts = {}


def get_font(path, size):
    if (path, size) not in _fonts:
        _fonts[(path, size)] = ImageFont.truetype(path, size, layout_engine=ImageFont.Layout.RAQM if HAS_RAQM else ImageFont.Layout.BASIC)
    return _fonts[(path, size)]


def is_arabic(text):
    """True when the text has Arabic letters. Arabic-Indic digits alone (a phone number, a date) do not count."""
    return re.search(r"[؀-ٟ٪-ۿ]", text) is not None


def arabic_font_needed(text):
    """Arabic letters or Arabic-Indic digits: Latin fonts have neither."""
    return is_arabic(text) or re.search(r"[٠-٩]", text) is not None


def prepare(text):
    """(text to draw, drawing options) so Arabic comes out joined and right to left."""
    if HAS_RAQM:
        return text, ({"direction": "rtl", "language": "ar"} if is_arabic(text) else {"direction": "ltr"})
    if is_arabic(text):
        import arabic_reshaper
        from bidi.algorithm import get_display
        return get_display(arabic_reshaper.reshape(text)), {}
    return text, {}


def draw_text(draw, x_left, x_right, y, text, font, color, align, stroke=0):
    shown, options = prepare(text)
    width = draw.textlength(shown, font=font, **options)
    x = {"right": x_right - width, "left": x_left, "center": (x_left + x_right - width) / 2}[align]
    draw.text((x, y), shown, font=font, fill=color, stroke_width=stroke, stroke_fill=color, **options)


def text_width(draw, text, font):
    shown, options = prepare(text)
    return draw.textlength(shown, font=font, **options)


def typeset_ltr(text):
    """A left-to-right mark before each separator keeps Arabic-Indic digit groups ("٠٥٠ ١٢٣ ٤٥٦٧") in reading order,
    as a typesetter would; drawn naively, any bidi engine reverses the groups. The marks never enter an answer."""
    if re.search(r"[٠-٩]", text) and not is_arabic(text):
        return re.sub(r"(?<=[٠-٩])(?=[\s\-/.:])", "‎", text)
    return text


def pick_font(rng, arabic, style):
    if style == "sign" and rng.random() < 0.6:
        return rng.choice(DISPLAY_FONTS["ar" if arabic else "en"])
    if not arabic:
        return rng.choice(ENGLISH_SPECIAL_FONTS if rng.random() < 0.15 and ENGLISH_SPECIAL_FONTS else ENGLISH_BODY_FONTS)
    return rng.choice(ARABIC_SPECIAL_FONTS if rng.random() < 0.15 and ARABIC_SPECIAL_FONTS else ARABIC_BODY_FONTS)


def colours(rng, style):
    if style == "page":
        return tuple(rng.randint(232, 255) for _ in range(3)), tuple([rng.randint(0, 60)] * 3)
    if style == "colour":
        return tuple(rng.randint(200, 255) for _ in range(3)), tuple(rng.randint(0, 110) for _ in range(3))
    if style == "slide" and rng.random() < 0.5:
        return tuple(rng.randint(225, 255) for _ in range(3)), rng.choice([(20, 40, 120), (150, 20, 20), (20, 90, 40), (30, 30, 30)])
    return tuple(rng.randint(0, 140) for _ in range(3)), tuple(rng.randint(225, 255) for _ in range(3))


def background(rng, size, colour):
    if rng.random() < 0.7:
        return Image.new("RGB", size, colour)
    w, h = size
    other = np.array([min(255, max(0, c + rng.randint(-50, 50))) for c in colour], dtype="float32")
    t = np.linspace(0, 1, w if rng.random() < 0.5 else h, dtype="float32")
    ramp = np.array(colour, dtype="float32")[None] * (1 - t[:, None]) + other[None] * t[:, None]
    grid = np.broadcast_to(ramp[None], (h, w, 3)) if len(t) == w else np.broadcast_to(ramp[:, None], (h, w, 3))
    return Image.fromarray(grid.astype("uint8"))


def render_lines(lines, rng, style=None, min_size=20):
    style = style or rng.choices(["page", "colour", "slide", "sign"], [0.6, 0.2, 0.15, 0.05])[0]
    arabic = any(is_arabic(l) for l in lines)
    path = pick_font(rng, arabic, style)
    size = rng.randint(30, 56) if style == "sign" else rng.randint(min_size, max(min_size, 46))
    probe = ImageDraw.Draw(Image.new("RGB", (10, 10)))
    widest = max(probe.textlength(prepare(l)[0], font=get_font(path, size), **prepare(l)[1]) for l in lines)
    if widest > 2100:
        size = max(min(min_size, 18), int(size * 2100 / widest))
        widest = max(probe.textlength(prepare(l)[0], font=get_font(path, size), **prepare(l)[1]) for l in lines)
    font = get_font(path, size)
    pad, line_height = rng.randint(10, 32), int(size * rng.uniform(1.45, 1.9))
    width = int(widest + 2 * pad + rng.randint(0, 80))
    bg, ink = colours(rng, style)
    img = background(rng, (width, 2 * pad + line_height * len(lines)), bg)
    draw = ImageDraw.Draw(img)
    stroke = 1 if rng.random() < (0.4 if style in ("slide", "sign") else 0.1) else 0
    centred = style in ("slide", "sign") and rng.random() < 0.4
    if style == "page" and rng.random() < 0.08:
        y = pad + line_height * len(lines) - int(size * 0.25)
        draw.line([(pad // 2, y), (width - pad // 2, y)], fill=tuple(min(255, c + 90) for c in ink), width=1)
    for i, line in enumerate(lines):
        align = "center" if centred else ("right" if is_arabic(line) else "left")
        draw_text(draw, pad, width - pad, pad + i * line_height, line, font, ink, align, stroke)
    return img


def perspective_coeffs(target, source):
    rows = []
    for (x, y), (u, v) in zip(target, source):
        rows.append([x, y, 1, 0, 0, 0, -u * x, -u * y])
        rows.append([0, 0, 0, x, y, 1, -v * x, -v * y])
    return np.linalg.solve(np.array(rows, dtype="float64"), np.array(source, dtype="float64").reshape(8)).tolist()


def perspective(img, rng, strength):
    w, h = img.size
    dx, dy = w * strength, h * strength
    target = [(rng.uniform(0, dx), rng.uniform(0, dy)), (w - rng.uniform(0, dx), rng.uniform(0, dy)),
              (w - rng.uniform(0, dx), h - rng.uniform(0, dy)), (rng.uniform(0, dx), h - rng.uniform(0, dy))]
    return img.transform((w, h), Image.PERSPECTIVE, perspective_coeffs(target, [(0, 0), (w, 0), (w, h), (0, h)]),
                         Image.BILINEAR, fillcolor=img.getpixel((0, 0)))


def damage(img, rng, level):
    if level == 0:
        return img
    if level >= 2 and rng.random() < 0.3:
        img = perspective(img, rng, 0.04)
    if rng.random() < 0.5 * level:
        img = img.rotate(rng.uniform(-1.5, 1.5) * level, expand=True, fillcolor=img.getpixel((0, 0)))
    if rng.random() < 0.5 * level:
        img = img.filter(ImageFilter.GaussianBlur(rng.uniform(0.3, 0.9) * level))
    if rng.random() < 0.4 * level:
        a = np.asarray(img).astype("float32")
        a += np.random.default_rng(rng.randint(0, 10**9)).normal(0, 6 * level, a.shape)
        img = Image.fromarray(a.clip(0, 255).astype("uint8"))
    if rng.random() < 0.4 * level:
        buf = io.BytesIO()
        img.save(buf, "JPEG", quality=rng.randint(30, 70))
        img = Image.open(io.BytesIO(buf.getvalue())).convert("RGB")
    if rng.random() < 0.3 * level:
        w, h = img.size
        s = rng.uniform(0.6, 0.85)
        img = img.resize((max(28, int(w * s)), max(28, int(h * s)))).resize((w, h))
    return img


def smooth_field(rng, shape, sigma, amplitude):
    """A smooth random displacement field (pixels), for wobble and elastic distortion."""
    noise = np.random.default_rng(rng.randint(0, 2**31)).uniform(-1, 1, shape).astype(np.float32)
    field = cv2.GaussianBlur(noise, (0, 0), sigma)
    return field / (np.abs(field).max() + 1e-6) * amplitude


def paper(rng, size, baseline=None, line_gap=None):
    """Off-white paper: a tint, soft blotches, fine grain, and sometimes ruled lines."""
    w, h = size
    tint = np.array(rng.choice([(250, 248, 240), (246, 242, 230), (252, 252, 250), (240, 236, 222), (236, 239, 243)]), dtype=np.float32)
    seed = rng.randint(0, 2**31)
    blotch = np.asarray(Image.fromarray((np.random.default_rng(seed).random((max(2, h // 24), max(2, w // 24))) * 255).astype(np.uint8))
                        .resize((w, h), Image.BICUBIC), dtype=np.float32)[..., None] / 255.0
    a = tint[None, None, :] * (0.93 + 0.07 * blotch) + np.random.default_rng(seed + 1).normal(0, 2.5, (h, w, 1))
    img = Image.fromarray(np.clip(a, 0, 255).astype(np.uint8))
    if baseline is not None and rng.random() < 0.35:
        d = ImageDraw.Draw(img)
        colour = rng.choice([(165, 185, 220), (185, 185, 185), (215, 165, 165)])
        for y in range(int(baseline) % int(line_gap), h, int(line_gap)):
            d.line([(0, y), (w, y)], fill=colour, width=1)
    return img


def render_handwriting(text, rng, language):
    """One line in a handwriting-like font, made to look written (see the section text)."""
    path = rng.choice(HAND_FONTS["ar" if language == "ar" else "en"])
    size = rng.randint(26, 46)
    probe = ImageDraw.Draw(Image.new("L", (10, 10)))
    width = text_width(probe, text, get_font(path, size))
    if width > 1900:
        size = max(18, int(size * 1900 / width))
        width = text_width(probe, text, get_font(path, size))
    font = get_font(path, size)
    shear = rng.uniform(-0.3, 0.3)
    pad_y = int(size * rng.uniform(0.5, 0.9))
    h = int(size * 1.9) + 2 * pad_y
    pad_x = int(rng.randint(12, 36) + abs(shear) * h / 2)
    w = int(width) + 2 * pad_x
    layer = Image.new("L", (w, h), 0)
    shown, options = prepare(text)
    ImageDraw.Draw(layer).text((pad_x, pad_y), shown, font=font, fill=255, **options)
    alpha = np.asarray(layer, dtype=np.float32) / 255.0
    pen = rng.random()
    if pen < 0.3:
        alpha = cv2.dilate(alpha, np.ones((2, 2), np.uint8))               # a thicker pen
    elif pen < 0.45 and size >= 30:
        alpha = cv2.erode(alpha, np.ones((2, 2), np.uint8))                # a finer pen
    ys, xs = np.mgrid[0:h, 0:w].astype(np.float32)
    wave = rng.uniform(0, size * 0.12) * np.sin(2 * np.pi * xs / rng.uniform(150, 600) + rng.uniform(0, 6.3))
    map_x = (xs + shear * (ys - h / 2) + smooth_field(rng, (h, w), rng.uniform(3, 6), rng.uniform(0, 1.6))).astype(np.float32)
    map_y = (ys + wave + smooth_field(rng, (h, w), rng.uniform(3, 6), rng.uniform(0, 1.6))).astype(np.float32)
    alpha = cv2.remap(alpha, map_x, map_y, cv2.INTER_LINEAR, borderValue=0)
    pressure = 0.75 + 0.25 * (smooth_field(rng, (h, w), 25, 1.0) + 1) / 2
    alpha = np.clip(alpha * pressure, 0, 1)[..., None]
    ink = np.array(rng.choice([(20, 30, 120), (15, 15, 20), (35, 35, 60), (10, 45, 95), (60, 25, 25)]), dtype=np.float32)
    bg = np.asarray(paper(rng, (w, h), baseline=pad_y + size * 1.15, line_gap=size * 1.9), dtype=np.float32)
    return Image.fromarray(np.clip(bg * (1 - alpha) + ink * alpha, 0, 255).astype(np.uint8))

## C. Augmentation for real handwriting, and contrast enhancement
Every real handwritten line is augmented when it is sampled (the model sees each line several times): a small
rotation, shear and scale; grid and elastic distortion; a thicker or thinner stroke; brightness and contrast; noise
and blur. This is the set behind the 8.46% KHATT result of CER-HV. `clahe(img)` is contrast-limited adaptive histogram
equalisation on the grey image, the step that helped faded manuscripts in the 2026 shared task; phase 1 measures
whether it helps ours.

In [ ]:
def grid_distort(a, rng, steps=5, limit=0.08):
    h, w = a.shape[:2]
    xs = np.linspace(0, w, steps + 1)
    ys = np.linspace(0, h, steps + 1)
    dx = (np.array([rng.uniform(-limit, limit) for _ in range(steps + 1)]) * w / steps).astype(np.float32)
    dy = (np.array([rng.uniform(-limit, limit) for _ in range(steps + 1)]) * h / steps).astype(np.float32)
    map_x = np.interp(np.arange(w), xs, xs + dx).astype(np.float32)[None, :].repeat(h, 0)
    map_y = np.interp(np.arange(h), ys, ys + dy).astype(np.float32)[:, None].repeat(w, 1)
    return map_x, map_y


def augment_line(img, rng):
    """A random mix of the CER-HV augmentations, mild enough that the text stays readable."""
    a = np.asarray(img.convert("RGB")).copy()
    h, w = a.shape[:2]
    border = tuple(int(v) for v in np.median(a.reshape(-1, 3), axis=0))
    if rng.random() < 0.5:                                                       # rotation, shear, scale
        m = cv2.getRotationMatrix2D((w / 2, h / 2), rng.uniform(-1.5, 1.5), rng.uniform(0.92, 1.06))
        m[0, 1] += rng.uniform(-0.15, 0.15)
        a = cv2.warpAffine(a, m, (w, h), borderValue=border)
    if rng.random() < 0.25:                                                      # grid distortion
        mx, my = grid_distort(a, rng)
        a = cv2.remap(a, mx, my, cv2.INTER_LINEAR, borderValue=border)
    if rng.random() < 0.25:                                                      # elastic distortion
        sigma, strength = rng.uniform(4, 7), rng.uniform(0.8, 2.0)
        ys, xs = np.mgrid[0:h, 0:w].astype(np.float32)
        a = cv2.remap(a, (xs + smooth_field(rng, (h, w), sigma, strength)).astype(np.float32),
                      (ys + smooth_field(rng, (h, w), sigma, strength)).astype(np.float32), cv2.INTER_LINEAR, borderValue=border)
    if rng.random() < 0.3:                                                       # thicker (erode dark ink grows) or thinner strokes
        a = (cv2.erode if rng.random() < 0.5 else cv2.dilate)(a, np.ones((2, 2), np.uint8))
    if rng.random() < 0.5:                                                       # brightness and contrast
        a = np.clip(a.astype(np.float32) * rng.uniform(0.75, 1.25) + rng.uniform(-25, 25), 0, 255).astype(np.uint8)
    if rng.random() < 0.3:                                                       # sensor noise
        a = np.clip(a + np.random.default_rng(rng.randint(0, 2**31)).normal(0, rng.uniform(3, 10), a.shape), 0, 255).astype(np.uint8)
    if rng.random() < 0.2:
        a = cv2.GaussianBlur(a, (0, 0), rng.uniform(0.4, 0.9))
    return Image.fromarray(a)


def grayscale(img):
    return img.convert("L").convert("RGB")


def clahe(img, clip=2.0, grid=8):
    g = np.asarray(img.convert("L"))
    return Image.fromarray(cv2.createCLAHE(clipLimit=clip, tileGridSize=(grid, grid)).apply(g)).convert("RGB")

## D. Loading the prepared data
Everything comes from the folder part 1b v6 made. Training sources are the `train_*` datasets, already filtered
against every test set (no shared 4-gram, no identical text, no near-copy image). Progress checks use validation
splits only. KHATT comes from the official release when part 1b found it, otherwise from the Teklia copy with each
repeated passage capped at 3 copies.

In [ ]:
from datasets import load_from_disk

DIACRITICS = re.compile(r"[ً-ْٰـ]")   # tashkeel, dagger alif, tatweel


def strip_marks(text):
    return DIACRITICS.sub("", text)


def read_text(name):
    return json.load(open(os.path.join(DATA_DIR, "text", name), encoding="utf-8"))


def split_pool(sentences):
    sentences = list(sentences)
    random.Random(0).shuffle(sentences)                 # the same split as v5 and as part 1b's form filler
    cut = max(1, len(sentences) // 20)
    return sentences[cut:] or sentences, sentences[:cut]   # (training pool, held-out pool)


_datasets = {}


def dataset(name):
    """A dataset part 1b saved (memory-mapped from the Colab disk), or None if it was not built."""
    if name not in _datasets:
        path = os.path.join(DATA_DIR, "datasets", name)
        _datasets[name] = load_from_disk(path) if os.path.exists(os.path.join(path, "state.json")) else None
    return _datasets[name]


data_start = time.time()
POOLS = {"msa": split_pool(read_text("wikipedia_ar.json")), "tashkeel": split_pool(read_text("tashkeel_ar.json")),
         "en": split_pool(read_text("wikipedia_en.json"))}
POOLS["classical"] = POOLS["tashkeel"]                  # the same sentences, drawn with their diacritics removed
VOCAB = json.load(open(os.path.join(DATA_DIR, "text", "vocab.json"), encoding="utf-8"))
TRAIN = {k: dataset(f"train_{k}") for k in ["khatt_official", "khatt_teklia", "muharaf", "memoir", "arams", "iam", "gnhk", "synthdog", "olmocr"]}
TRAIN["forms_template"] = dataset("forms_template_train")
SELF_LABELLED_DIR = os.path.join(BASE_DIR, f"{RUN_NAME}_self_labelled")
TRAIN["self_labelled"] = load_from_disk(SELF_LABELLED_DIR) if os.path.exists(os.path.join(SELF_LABELLED_DIR, "state.json")) else None
KHATT_TRAIN = "khatt_official" if TRAIN["khatt_official"] is not None else "khatt_teklia"
VAL = {"khatt": dataset("khatt_official_validation") or dataset("val_khatt_teklia"), "muharaf": dataset("muharaf_validation"),
       "memoir": dataset("val_memoir"), "arams": dataset("arams_val"), "iam": dataset("iam_validation"), "gnhk": dataset("val_gnhk"),
       "synthdog": dataset("val_synthdog"), "forms_template": dataset("val_forms_template")}
for name in ["msa", "tashkeel", "en"]:
    print(f"{name:9s} sentences: {len(POOLS[name][0]):,} train / {len(POOLS[name][1]):,} held out")
print("training sources:", ", ".join(f"{k} {len(v):,}" for k, v in TRAIN.items() if v is not None))
print("not available:   ", ", ".join(k for k, v in TRAIN.items() if v is None) or "none")
print("validation sets: ", ", ".join(f"{k} {len(v):,}" for k, v in VAL.items() if v is not None))
print(f"KHATT for training: {KHATT_TRAIN} | data ready in {(time.time() - data_start) / 60:.1f} min")

## E. Training examples
Every example is a triple **(image, instruction, answer)**. The v5 generators are unchanged (drawn lines, bilingual
lines, scene words, v5 forms), and v6 adds:

| Function | Image | Answer |
| --- | --- | --- |
| `handwriting_style_example` | a line drawn by `render_handwriting` (section B) | the text |
| `line_source(name)` | a real line from a training source, augmented (section C), optionally grey or CLAHE | its transcription |
| `page_source(name)` | a SynthDoG or olmOCR page | its text |
| `form_v2_example` | a form from generator v2: 4–10 of 40 field types, 46 layouts (6 label arrangements, 5 value styles, 1 or 2 columns), section headers, tick boxes, item tables, stamps; values typed, in handwriting fonts, or pasted real handwriting; empty fields that must come back as `null`; key order shuffled half the time | the fields as JSON |
| `template_form_example` | a training template filled by part 1b (training fonts and writers only) | the fields as JSON |

AraMS-28k lines are trained with their own instruction (`OCR_LETTERS_INSTRUCTION`), because their transcriptions keep
only Arabic letters; every evaluation uses the usual instruction for every model. `Mixture` combines sources by
weight; a source that is not available is dropped and its share spread over the rest, and `describe()` prints the
shares actually used.

In [ ]:
TO_ARABIC_DIGITS = str.maketrans("0123456789", "٠١٢٣٤٥٦٧٨٩")
TO_WESTERN_DIGITS = str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")
OCR_INSTRUCTION = "Read all the text in the image exactly as written. اقرأ النص كما هو مكتوب."
OCR_PLAIN_INSTRUCTION = "Read all the text in the image. Write Arabic without diacritics. اقرأ النص بدون تشكيل."
OCR_LETTERS_INSTRUCTION = "Read the Arabic letters in the image. Write no diacritics, digits or punctuation. اقرأ الحروف فقط بدون تشكيل أو أرقام أو ترقيم."


def words_in_line(rng, block=False):
    if block:
        return rng.randint(3, 10)
    x = rng.random()
    if x < 0.08:
        return rng.randint(1, 2)
    if x < 0.45:
        return rng.randint(3, 8)
    if x < 0.80:
        return rng.randint(9, 14)
    return rng.randint(15, 20)


def text_piece(rng, pool, n):
    words = rng.choice(pool).split()
    while len(words) < n:
        words += rng.choice(pool).split()
    start = rng.randint(0, len(words) - n)
    return " ".join(words[start:start + n])


LATIN_EXTRAS = ["PDF", "Invoice INV-2026", "ID", "Email", "UAE", "AED 250", "Ref A-17", "www.example.ae", "VAT 5%", "Dubai Mall",
                "ERP", "Microsoft Teams", "Etihad", "ADNOC", "Block B-12", "iPhone 15", "Wi-Fi", "QR", "GPS", "COVID-19"]


def make_line(rng, source, heldout, n, extras):
    text = text_piece(rng, POOLS[source][1 if heldout else 0], n)
    if source == "classical":
        text = strip_marks(text)
    if source != "en" and extras and rng.random() < 0.25:
        number = str(rng.choice([rng.randint(1, 99), rng.randint(100, 99999), rng.randint(1900, 2030)]))
        text += " " + (number.translate(TO_ARABIC_DIGITS) if rng.random() < 0.6 else number)
    if source != "en" and extras and rng.random() < 0.1:
        text += " " + rng.choice(LATIN_EXTRAS)
    return text


def text_line_example(rng, language, damage_level, max_lines=1, heldout=False, source=None, style=None):
    source = "en" if language == "en" else (source or "msa")
    count = rng.randint(2, max_lines) if max_lines > 1 else 1
    lines = [make_line(rng, source, heldout, words_in_line(rng, block=count > 1), extras=damage_level > 0) for _ in range(count)]
    image = damage(render_lines(lines, rng, style, min_size=28 if damage_level == 2 else 20), rng, damage_level)
    return image, OCR_PLAIN_INSTRUCTION if source == "classical" else OCR_INSTRUCTION, "\n".join(lines)


def scene_word_example(rng, heldout=False):
    language = "en" if rng.random() < 0.15 else "ar"
    source = "en" if language == "en" else rng.choice(["msa", "msa", "classical"])
    text = make_line(rng, source, heldout, rng.randint(1, 3), extras=False)
    img = render_lines([text], rng, style=rng.choice(["sign", "sign", "colour", "slide"]))
    img = img.rotate(rng.uniform(-4, 4), expand=True, fillcolor=img.getpixel((0, 0)))
    return damage(img, rng, rng.choice([1, 2])), OCR_PLAIN_INSTRUCTION if language == "ar" else OCR_INSTRUCTION, text


def handwriting_style_example(rng, heldout=False):
    """A rendered handwriting-style line (phase 2 pre-training): 80% Arabic, 20% English."""
    language = "en" if rng.random() < 0.2 else "ar"
    source = "en" if language == "en" else rng.choice(["msa", "msa", "classical"])
    text = make_line(rng, source, heldout, rng.randint(2, 12), extras=rng.random() < 0.3)
    img = damage(render_handwriting(text, rng, language), rng, rng.choice([0, 0, 1]))
    return img, OCR_PLAIN_INSTRUCTION if source == "classical" else OCR_INSTRUCTION, text


def line_source(name, instruction=OCR_INSTRUCTION, augment=True, look=None):
    """Examples from a line dataset (a key of TRAIN, or any saved dataset). look: None, 'gray' or 'clahe'."""
    ds = TRAIN.get(name) if name in TRAIN else dataset(name)
    if ds is None or len(ds) == 0:
        return None

    def example(rng):
        row = ds[rng.randrange(len(ds))]
        img = row["image"].convert("RGB")
        img = grayscale(img) if look == "gray" else clahe(img) if look == "clahe" else img
        if augment:
            if rng.random() < 0.8:
                img = augment_line(img, rng)
            img = damage(img, rng, rng.choice([0, 0, 1]))
        return img, instruction, row["text"]
    return example


def page_source(name):
    ds = TRAIN.get(name)
    if ds is None or len(ds) == 0:
        return None

    def example(rng):
        row = ds[rng.randrange(len(ds))]
        return damage(row["image"].convert("RGB"), rng, rng.choice([0, 1])), OCR_INSTRUCTION, row["text"]
    return example


class Mixture:
    """Draws an example from one of several sources, chosen by weight. Unavailable sources (None) are dropped."""

    def __init__(self, name, parts):
        self.name = name
        available = [(w, label, fn) for w, label, fn in parts if fn is not None and w > 0]
        self.missing = [label for w, label, fn in parts if fn is None and w > 0]
        assert available, f"{name}: no source is available"
        total = sum(w for w, _, _ in available)
        self.parts = [(w / total, label, fn) for w, label, fn in available]

    def __call__(self, rng):
        x, acc = rng.random(), 0.0
        for w, _, fn in self.parts:
            acc += w
            if x < acc:
                return fn(rng)
        return self.parts[-1][2](rng)

    def shares(self):
        return {label: round(w, 4) for w, label, _ in self.parts}

    def describe(self):
        text = f"{self.name}:\n" + "\n".join(f"   {100 * w:5.1f}%  {label}" for w, label, _ in self.parts)
        return text + (f"\n   (not available, share spread over the rest: {', '.join(self.missing)})" if self.missing else "")


class GeneratedData(torch.utils.data.Dataset):
    """n examples; example i is made by recipe(random.Random(seed, offset + i)), so a resumed run continues with
    exactly the examples it would have seen."""

    def __init__(self, recipe, n, seed, offset=0):
        self.recipe, self.n, self.seed, self.offset = recipe, n, seed, offset

    def __len__(self):
        return self.n

    def __getitem__(self, i):
        return self.recipe(random.Random(self.seed * 1_000_003 + self.offset + i))


def show_examples(recipe, n=4):
    for i in range(n):
        img, instruction, answer = recipe(random.Random(99 + i))
        print(f"{img.size[0]}x{img.size[1]} px -> {answer[:110]!r}")
    return recipe(random.Random(99))[0]

### The v5 form generator and bilingual lines
Kept as they were in v5 (only the e-mail and web addresses now use reserved example domains): the bilingual lines are
part of the reading mix, and the v5 forms stay as a regression set ("Generated forms (v5 generator)") and as 15% of
the forms adapter's data.

In [ ]:
MALE_NAMES = ["محمد", "أحمد", "خالد", "سعيد", "عبدالله", "راشد", "حمدان", "منصور", "سلطان", "علي", "حمد", "يوسف", "عمر", "فيصل"]
FEMALE_NAMES = ["فاطمة", "مريم", "عائشة", "شيخة", "نورة", "سارة", "ليلى", "موزة", "حصة", "آمنة", "هند", "لطيفة"]
FAMILY_NAMES = ["الكعبي", "المنصوري", "الظاهري", "النعيمي", "الشامسي", "المزروعي", "الكتبي", "السويدي", "المهيري", "الهاشمي", "البلوشي", "العامري"]
LATIN_NAMES = {"محمد": "Mohammed", "أحمد": "Ahmed", "خالد": "Khalid", "سعيد": "Saeed", "عبدالله": "Abdulla", "راشد": "Rashid",
               "حمدان": "Hamdan", "منصور": "Mansoor", "سلطان": "Sultan", "علي": "Ali", "حمد": "Hamad", "يوسف": "Yousef", "عمر": "Omar",
               "فيصل": "Faisal", "فاطمة": "Fatima", "مريم": "Mariam", "عائشة": "Aisha", "شيخة": "Shaikha", "نورة": "Noura", "سارة": "Sara",
               "ليلى": "Laila", "موزة": "Moza", "حصة": "Hessa", "آمنة": "Amna", "هند": "Hind", "لطيفة": "Latifa",
               "الكعبي": "Al Kaabi", "المنصوري": "Al Mansoori", "الظاهري": "Al Dhaheri", "النعيمي": "Al Nuaimi", "الشامسي": "Al Shamsi",
               "المزروعي": "Al Mazrouei", "الكتبي": "Al Ketbi", "السويدي": "Al Suwaidi", "المهيري": "Al Muhairi", "الهاشمي": "Al Hashemi",
               "البلوشي": "Al Balushi", "العامري": "Al Ameri"}
EMIRATES = ["أبوظبي", "دبي", "الشارقة", "عجمان", "أم القيوين", "رأس الخيمة", "الفجيرة"]
NATIONALITIES = ["الإمارات", "السعودية", "مصر", "الأردن", "الهند", "باكستان", "الفلبين", "سوريا", "لبنان", "المغرب", "بنغلاديش", "السودان"]
COMPANY_WORDS = {"النور": "Al Noor", "الخليج": "Al Khaleej", "الإمارات": "Emirates", "المستقبل": "Al Mustaqbal", "البركة": "Al Baraka",
                 "الصقر": "Al Saqr", "النخيل": "Al Nakheel", "الأفق": "Al Ufuq", "السلام": "Al Salam", "الواحة": "Al Waha"}
COMPANY_KINDS = [("للتجارة", "Trading LLC"), ("للمقاولات", "Contracting LLC"), ("للخدمات", "Services LLC"), ("للتقنية", "Technologies FZE")]
STREETS = ["شارع الكورنيش", "شارع الشيخ زايد", "شارع المطار", "شارع خليفة", "شارع حمدان", "شارع الوحدة"]
FORM_TITLES = [("استمارة طلب خدمة", "Service Request Form"), ("فاتورة", "Invoice"), ("طلب تسجيل", "Registration Application"),
               ("طلب تجديد رخصة", "Licence Renewal Request"), ("إيصال دفع", "Payment Receipt"), ("نموذج بيانات الموظف", "Employee Data Form")]
FIELD_LABELS = {
    "reference_number": ("رقم الطلب", "Reference No."), "date": ("التاريخ", "Date"), "full_name": ("الاسم الكامل", "Full Name"),
    "phone": ("رقم الهاتف", "Phone"), "emirate": ("الإمارة", "Emirate"), "amount_aed": ("المبلغ (درهم)", "Amount (AED)"),
    "emirates_id": ("رقم الهوية", "Emirates ID"), "nationality": ("الجنسية", "Nationality"), "email": ("البريد الإلكتروني", "Email"),
    "expiry_date": ("تاريخ الانتهاء", "Expiry Date"), "company": ("اسم الشركة", "Company"), "address": ("العنوان", "Address"),
}
CORE_FIELDS = ["reference_number", "date", "full_name", "phone", "emirate", "amount_aed"]


def form_instruction(keys):
    """v5's form instruction (used for the v5 generator's forms, so v5 checkpoints see the wording they trained on)."""
    return ("Extract the fields from this form as JSON with keys: " + ", ".join(keys)
            + ". Use Western digits, dates as YYYY-MM-DD and phone numbers as digits only.")


def form_instruction_v6(keys):
    """v6 adds null for empty fields (generator v2, the template forms, the held-out forms, FUNSD)."""
    return ("Extract the fields from this form as JSON with keys: " + ", ".join(keys)
            + ". Use Western digits, dates as YYYY-MM-DD, phone numbers as digits only, and null for an empty field.")


def random_name(rng):
    first, middle, family = rng.choice(MALE_NAMES + FEMALE_NAMES), rng.choice(MALE_NAMES), rng.choice(FAMILY_NAMES)
    return f"{first} {middle} {family}", f"{LATIN_NAMES[first]} {LATIN_NAMES[middle]} {LATIN_NAMES[family]}"


def random_date(rng, year_from=2020, year_to=2027):
    year, month, day = rng.randint(year_from, year_to), rng.randint(1, 12), rng.randint(1, 28)
    answer = f"{year:04d}-{month:02d}-{day:02d}"
    printed = rng.choice([f"{day:02d}/{month:02d}/{year}", f"{day:02d}-{month:02d}-{year}", f"{day:02d}.{month:02d}.{year}", answer])
    return answer, printed


def field_value(rng, key, show):
    if key == "reference_number":
        prefix, number = rng.choice(["SR", "INV", "REG", "REQ", "PAY"]), str(rng.randint(10000, 999999))
        return f"{prefix}-{number}", f"{prefix}-{show(number)}"
    if key in ("date", "expiry_date"):
        answer, printed = random_date(rng, *((2025, 2031) if key == "expiry_date" else (2020, 2027)))
        return answer, show(printed)
    if key == "full_name":
        arabic, latin = random_name(rng)
        return arabic, arabic
    if key == "phone":
        digits = "05" + "".join(str(rng.randint(0, 9)) for _ in range(8))
        printed = rng.choice([f"{digits[:3]} {digits[3:6]} {digits[6:]}", digits, f"+971 {digits[1:3]} {digits[3:6]} {digits[6:]}"])
        return digits, (show(printed) if rng.random() < 0.3 else printed)
    if key == "emirate":
        value = rng.choice(EMIRATES)
        return value, value
    if key == "amount_aed":
        amount = rng.randint(50, 50000) + rng.choice([0, 0.25, 0.5, 0.75])
        printed = rng.choice([f"{amount:,.2f}", f"AED {amount:,.2f}", f"{amount:,.2f} درهم"])
        return f"{amount:.2f}", show(printed)
    if key == "emirates_id":
        answer = f"784-{rng.randint(1950, 2010)}-{rng.randint(1000000, 9999999)}-{rng.randint(0, 9)}"
        return answer, answer
    if key == "nationality":
        value = rng.choice(NATIONALITIES)
        return value, value
    if key == "email":
        arabic, latin = random_name(rng)
        parts = latin.lower().replace("al ", "al").split()
        value = rng.choice([f"{parts[0]}.{parts[-1]}", f"{parts[0][0]}{parts[-1]}", f"{parts[0]}{rng.randint(1, 99)}"]) + "@" + rng.choice(["example.ae", "example.com"])
        return value, value
    if key == "company":
        arabic_kind, english_kind = rng.choice(COMPANY_KINDS)
        value = f"شركة {rng.choice(list(COMPANY_WORDS))} {arabic_kind}"
        return value, value
    if key == "address":
        value = f"{rng.choice(EMIRATES)}، {rng.choice(STREETS)}، مبنى {rng.randint(1, 60)}"
        return value, show(value)
    raise KeyError(key)


def render_label_value(rng, label, value, style="page", min_size=22):
    path = pick_font(rng, True, style)
    size = rng.randint(min_size, 40)
    font = get_font(path, size)
    probe = ImageDraw.Draw(Image.new("RGB", (10, 10)))
    value_font = font if arabic_font_needed(value) else get_font(rng.choice(MONO_FONTS + ENGLISH_BODY_FONTS), size)
    shown_value = typeset_ltr(value)
    label_w, value_w = text_width(probe, label, font), text_width(probe, shown_value, value_font)
    pad, gap = rng.randint(10, 32), int(size * 0.5)
    width = int(label_w + value_w + gap + 2 * pad + rng.randint(0, 60))
    bg, ink = colours(rng, style)
    img = background(rng, (width, 2 * pad + int(size * 1.7)), bg)
    draw = ImageDraw.Draw(img)
    draw_text(draw, pad, width - pad, pad, label, font, ink, "right")
    draw_text(draw, pad, width - pad - int(label_w) - gap, pad, shown_value, value_font, ink, "right")
    return img


def bilingual_line_example(rng, heldout=False):
    kind = rng.random()
    show = (lambda s: s.translate(TO_ARABIC_DIGITS)) if rng.random() < 0.4 else (lambda s: s)
    if kind < 0.45:
        key = rng.choice(["reference_number", "phone", "emirates_id", "email", "amount_aed", "date", "full_name"])
        answer, printed = field_value(rng, key, show)
        if key == "full_name" and rng.random() < 0.7:
            printed = random_name(rng)[1]
        label = FIELD_LABELS[key][0] + ":"
        text = f"{label} {printed}"
        if rng.random() < 0.5:
            img = damage(render_label_value(rng, label, printed, rng.choice(["page", "page", "colour"])), rng, rng.choice([0, 1, 1]))
            return img, OCR_INSTRUCTION, text
    elif kind < 0.65:
        key = rng.choice(list(FIELD_LABELS))
        text = f"{FIELD_LABELS[key][0]} / {FIELD_LABELS[key][1]}"
    elif kind < 0.8:
        if rng.random() < 0.5:
            arabic_kind, english_kind = rng.choice(COMPANY_KINDS)
            word = rng.choice(list(COMPANY_WORDS))
            text = f"شركة {word} {arabic_kind} - {COMPANY_WORDS[word]} {english_kind}"
        else:
            text = rng.choice(["أبوظبي - Abu Dhabi", "دبي - Dubai", "الشارقة - Sharjah", "عجمان - Ajman", "رأس الخيمة - Ras Al Khaimah",
                               "الفجيرة - Fujairah", "أم القيوين - Umm Al Quwain", "العين - Al Ain"])
    else:
        words = make_line(rng, "msa", heldout, rng.randint(5, 12), extras=False).split()
        words.insert(rng.randint(1, len(words) - 1), rng.choice(LATIN_EXTRAS))
        text = " ".join(words)
    img = damage(render_lines([text], rng, rng.choice(["page", "page", "colour"]), min_size=22), rng, rng.choice([0, 1, 1]))
    return img, OCR_INSTRUCTION, text


def fit_text(draw, x_left, x_right, y, text, font, color, align):
    size = font.size
    while text_width(draw, text, font) > x_right - x_left and size > 14:
        size -= 2
        font = get_font(font.path, size)
    draw_text(draw, x_left, x_right, y, text, font, color, align)


def draw_stamp(draw, rng, box, text):
    x0, y0, x1, y1 = box
    colour = rng.choice([(150, 30, 30), (30, 60, 150), (40, 110, 60)])
    draw.ellipse(box, outline=colour, width=3)
    draw.ellipse((x0 + 8, y0 + 8, x1 - 8, y1 - 8), outline=colour, width=1)
    draw_text(draw, x0, x1, (y0 + y1) // 2 - (y1 - y0) // 10, text, get_font(rng.choice(ARABIC_BODY_FONTS), max(14, (y1 - y0) // 6)), colour, "center")


def draw_signature(draw, rng, box):
    x0, y0, x1, y1 = box
    points = sorted((rng.uniform(x0, x1), rng.uniform(y0, y1)) for _ in range(rng.randint(6, 12)))
    draw.line(points, fill=(20, 20, 90), width=2, joint="curve")


def draw_barcode(draw, rng, box):
    x0, y0, x1, y1 = box
    x = x0
    while x < x1:
        w = rng.choice([1, 2, 3])
        if rng.random() < 0.5:
            draw.rectangle((x, y0, x + w, y1), fill=(0, 0, 0))
        x += w + rng.choice([1, 2])


def form_example(rng, damage_level=1):
    """The v5 form generator, unchanged."""
    show = (lambda s: s.translate(TO_ARABIC_DIGITS)) if rng.random() < 0.5 else (lambda s: s)
    extra = [k for k in FIELD_LABELS if k not in CORE_FIELDS]
    keys = rng.sample(CORE_FIELDS, rng.randint(3, 6)) + rng.sample(extra, rng.randint(0, 3))
    rng.shuffle(keys)
    keys = keys[:rng.randint(4, 8)] if len(keys) > 4 else keys
    values = {k: field_value(rng, k, show) for k in keys}
    answer = {k: values[k][0] for k in keys}
    layout = rng.choice(["rows", "rows", "two_columns", "boxed", "table"])
    size = rng.randint(22, 32)
    arabic_font = get_font(rng.choice(ARABIC_BODY_FONTS), size)
    english_font = get_font(rng.choice(ENGLISH_BODY_FONTS), int(size * 0.8))
    value_fonts = {"ar": get_font(rng.choice(HAND_FONTS["ar"]), size) if rng.random() < 0.3 else arabic_font,
                   "en": get_font(rng.choice(HAND_FONTS["en"] if rng.random() < 0.3 else MONO_FONTS), size)}
    width, pad, row_height = rng.randint(700, 1200), rng.randint(24, 40), int(size * 2.1)
    rows = math.ceil(len(keys) / 2) if layout == "two_columns" else len(keys)
    header = 2 * row_height + (row_height if rng.random() < 0.5 else 0)
    footer = int(row_height * 2.5) if rng.random() < 0.6 else row_height // 2
    img = Image.new("RGB", (width, 2 * pad + header + row_height * rows + footer), tuple(rng.randint(238, 255) for _ in range(3)))
    draw = ImageDraw.Draw(img)
    title_ar, title_en = rng.choice(FORM_TITLES)
    if rng.random() < 0.4:
        draw.rectangle((pad, pad, pad + size * 2, pad + size * 2), outline=(120, 120, 120), width=2)
    draw_text(draw, pad, width - pad, pad, title_ar, get_font(arabic_font.path, size + 6), (0, 0, 0), "right")
    draw_text(draw, pad + size * 3, width - pad, pad + size // 2, title_en, english_font, (0, 0, 0), "left")
    y = pad + 2 * row_height
    if header > 2 * row_height:
        options = rng.choice([("جديد", "تجديد", "إلغاء"), ("فرد", "شركة", "حكومة")])
        ticked, x = rng.randint(0, 2), width - pad
        for i, option in enumerate(options):
            box = (x - size, y + 4, x, y + 4 + size)
            draw.rectangle(box, outline=(60, 60, 60), width=2)
            if i == ticked:
                draw.line([(box[0] + 4, box[1] + size // 2), (box[0] + size // 2, box[3] - 4), (box[2] - 3, box[1] + 3)], fill=(0, 0, 0), width=3)
            draw_text(draw, pad, x - size - 10, y, option, arabic_font, (40, 40, 40), "right")
            x -= size * 6
        y += row_height
    ink = (0, 0, 0)
    typeset = rng.random() < 0.5
    for i, key in enumerate(keys):
        label_ar, label_en = FIELD_LABELS[key]
        printed = values[key][1]
        value_font = value_fonts["ar" if arabic_font_needed(printed) else "en"]
        if typeset:
            printed = typeset_ltr(printed)
        label_ar_w = text_width(draw, label_ar + ":", arabic_font)
        if layout == "two_columns":
            x_left, x_right = (pad, width // 2 - pad // 2) if i % 2 == 1 else (width // 2 + pad // 2, width - pad)
            row_y = y + (i // 2) * row_height
            draw_text(draw, x_left, x_right, row_y, label_ar + ":", arabic_font, (40, 40, 40), "right")
            fit_text(draw, x_left, x_right - int(label_ar_w) - 12, row_y, printed, value_font, ink, "right")
            continue
        row_y = y + i * row_height
        if layout == "table":
            divider = width - pad - int(label_ar_w) - 24
            draw.rectangle((pad, row_y - 6, width - pad, row_y + row_height - 6), outline=(140, 140, 140), width=1)
            draw.line([(divider, row_y - 6), (divider, row_y + row_height - 6)], fill=(140, 140, 140), width=1)
        elif layout == "rows":
            draw.line([(pad, row_y - 6), (width - pad, row_y - 6)], fill=(170, 170, 170), width=1)
        draw_text(draw, pad, width - pad, row_y, label_ar + ":", arabic_font, (40, 40, 40), "right")
        area_left, area_right = pad, width - pad - int(label_ar_w) - 24
        if layout != "table":
            draw_text(draw, pad, width - pad, row_y, label_en, english_font, (90, 90, 90), "left")
            area_left = pad + int(text_width(draw, label_en, english_font)) + 24
        if layout == "boxed":
            box = (area_left, row_y - 4, area_right, row_y + row_height - 10)
            draw.rectangle(box, outline=(90, 90, 90), width=2)
            fit_text(draw, box[0] + 8, box[2] - 8, row_y, printed, value_font, ink, "center")
        else:
            fit_text(draw, area_left, area_right, row_y, printed, value_font, ink, "center")
    y_footer = y + rows * row_height + 6
    if footer > row_height:
        if rng.random() < 0.6:
            draw_stamp(draw, rng, (pad, y_footer, pad + int(row_height * 2.2), y_footer + int(row_height * 2.2)), rng.choice(["مصدق", "مدفوع", "مستلم"]))
        if rng.random() < 0.6:
            draw_signature(draw, rng, (width // 2 - row_height * 2, y_footer, width // 2 + row_height * 2, y_footer + row_height * 2))
        if rng.random() < 0.5:
            draw_barcode(draw, rng, (width - pad - row_height * 4, y_footer + row_height // 2, width - pad, y_footer + row_height * 2))
    return damage(img, rng, damage_level), form_instruction(keys), json.dumps(answer, ensure_ascii=False)

### Form generator v2
v5's generator had 12 field types, 4 layouts and about 38 names; v4 scored 1.000 on its own generator's forms and
0.149 on v5's, so a generator's own forms say little. v2 widens everything the roadmap lists:

- **40 field types** with several Arabic and English labels each (from `vocab.json`), 4–10 per form;
- **46 layouts**: Arabic label right and English left, Arabic only, English only, labels stacked over the value,
  labels above a box, or a two-column table; values on an underline, a dotted line, in a box, in character cells
  (combs) or on plain paper; one or two columns; plus section headers, a page border, a coloured title band, a
  logo box, decorative tick boxes and item tables, stamps, signatures and barcodes;
- **thousands of names**: random combinations of 194 Arabic first names, 100 family names, 104 English first names
  and 109 surnames;
- **handwriting**: values in handwriting fonts, or (for free text and addresses) real handwritten lines from the
  training writers pasted in;
- **empty fields** that must come back as `null`, and **key order** shuffled half the time.

Two-page forms are left out: the model takes one image per question here, and a stacked two-page image shrinks the
text below what 1,024 visual tokens can resolve. Every design is fictional; no real organisation's form is copied.

In [ ]:
FIELD_TYPES = VOCAB["field_types"]
FIELD_KIND = {k: v["kind"] for k, v in FIELD_TYPES.items()}


# --- fictional values: the same generator as part 1b's template filler ----------------------------------------------
def person(rng):
    first = rng.choice(VOCAB["arabic_female"] if rng.random() < 0.5 else VOCAB["arabic_male"])
    father, family = rng.choice(VOCAB["arabic_male"]), rng.choice(VOCAB["arabic_family"])
    return f"{first[0]} {father[0]} {family[0]}", f"{first[1]} {father[1]} {family[1]}"


def english_person(rng):
    return f"{rng.choice(VOCAB['english_first'])} {rng.choice(VOCAB['english_last'])}"


def a_date(rng, years):
    year, month, day = rng.randint(*years), rng.randint(1, 12), rng.randint(1, 28)
    iso = f"{year:04d}-{month:02d}-{day:02d}"
    shown = rng.choice([f"{day:02d}/{month:02d}/{year}", f"{day:02d}-{month:02d}-{year}", f"{day:02d}.{month:02d}.{year}", iso,
                        f"{year}/{month:02d}/{day:02d}", f"{day} {VOCAB['months_en'][month - 1]} {year}",
                        f"{day} {VOCAB['months_ar'][month - 1]} {year}"])
    return iso, shown


def make_value(kind, rng, script="ar", sentences=None):
    """(answer, text drawn on the form): Western digits, dates as YYYY-MM-DD, phone numbers as digits only."""
    show = (lambda s: s.translate(TO_ARABIC_DIGITS)) if (script == "ar" and rng.random() < 0.4) else (lambda s: s)
    pick = lambda pool: rng.choice(VOCAB[pool])[0 if script == "ar" else 1]
    if kind == "name_ar":
        value = person(rng)[0] if script == "ar" else english_person(rng)
        return value, value
    if kind == "name_en":
        value = person(rng)[1] if rng.random() < 0.6 else english_person(rng)
        return value, value
    if kind in ("date", "dob", "past_date", "future_date"):
        years = {"date": (2020, 2027), "dob": (1950, 2008), "past_date": (2015, 2026), "future_date": (2026, 2034)}[kind]
        iso, shown = a_date(rng, years)
        return iso, shown if re.search(r"[A-Za-z]", shown) else show(shown)            # no Arabic-Indic digits beside an English month
    if kind in ("phone", "mobile"):
        digits = ("05" + str(rng.choice([0, 2, 4, 5, 6, 8])) if kind == "mobile" else "0" + str(rng.choice([2, 3, 4, 6, 7, 9])))
        digits += "".join(str(rng.randint(0, 9)) for _ in range(7))
        spaced = f"{digits[:3]} {digits[3:6]} {digits[6:]}" if kind == "mobile" else f"{digits[:2]} {digits[2:5]} {digits[5:]}"
        shown = rng.choice([digits, spaced, "+971 " + spaced[1:], f"{digits[:3]}-{digits[3:]}"])
        return digits, show(shown) if rng.random() < 0.3 else shown
    if kind == "email":
        parts = (person(rng)[1] if rng.random() < 0.6 else english_person(rng)).lower().replace("al ", "al").split()
        value = rng.choice([f"{parts[0]}.{parts[-1]}", f"{parts[0][0]}{parts[-1]}", f"{parts[0]}{rng.randint(1, 99)}"]) + "@" + rng.choice(VOCAB["email_domains"])
        return value, value
    if kind in ("po_box", "quantity"):
        number = str(rng.randint(100, 999999) if kind == "po_box" else rng.randint(1, 500))
        return number, show(number)
    if kind in ("emirate", "city", "area", "nationality", "gender", "marital", "occupation", "relationship", "grade"):
        pool = {"emirate": "emirates", "area": "areas", "nationality": "nationalities", "marital": "marital",
                "occupation": "occupations", "relationship": "relationships", "grade": "grades", "gender": "gender"}.get(kind)
        value = pick(pool) if pool else rng.choice(VOCAB["emirates"] + VOCAB["cities"])[0 if script == "ar" else 1]
        return value, value
    if kind == "address":
        (ea, ee), (sa, se), n = rng.choice(VOCAB["emirates"]), rng.choice(VOCAB["streets"]), rng.randint(1, 120)
        value = f"{ea}، {sa}، مبنى {n}" if script == "ar" else f"Building {n}, {se}, {ee}"
        return value, show(value)
    if kind == "company":
        (wa, we), (ka, ke) = rng.choice(VOCAB["company_words"]), rng.choice(VOCAB["company_kinds"])
        value = f"شركة {wa} {ka}" if script == "ar" else f"{we} {ke}"
        return value, value
    if kind == "school":
        wa, we = rng.choice(VOCAB["school_words"])
        value = f"مدرسة {wa}" if script == "ar" else f"{we} School"
        return value, value
    if kind in ("reference", "invoice", "licence"):
        prefix = rng.choice({"reference": ["SR", "REQ", "REF", "APP"], "invoice": ["INV", "TAX-INV", "IN"], "licence": ["TL", "CN", "LIC"]}[kind])
        number = str(rng.randint(10_000, 9_999_999))
        return f"{prefix}-{number}", f"{prefix}-{show(number)}"
    if kind == "emirates_id":
        value = f"784-{rng.randint(1950, 2010)}-{rng.randint(1_000_000, 9_999_999)}-{rng.randint(0, 9)}"
        return value, value
    if kind == "passport":
        value = rng.choice("ABCDEFGHJKLMNPRSTUVWXYZ") + str(rng.randint(1_000_000, 99_999_999))
        return value, value
    if kind in ("amount", "amount_small"):
        amount = (rng.randint(50, 50_000) if kind == "amount" else rng.randint(1, 2_500)) + rng.choice([0, 0.25, 0.5, 0.75])
        shown = rng.choice([f"{amount:,.2f}", f"AED {amount:,.2f}", f"{amount:,.2f} درهم", f"{amount:.2f}"])
        return f"{amount:.2f}", show(shown)
    if kind == "plate":
        value = f"{rng.choice('ABCDEFGHIJKLMNOPQRS')} {rng.randint(1, 99_999)}"
        return value, value
    if kind == "choice":
        value = rng.choice(rng.choice(VOCAB["choices"]))[0 if script == "ar" else 1]
        return value, value
    words = rng.choice(sentences or ["ملاحظات عامة حول الطلب"]).split()
    start = rng.randint(0, max(0, len(words) - 4))
    value = " ".join(words[start:start + rng.randint(3, 8)])
    return value, show(value)


# --- real handwriting pasted into forms (training writers only) --------------------------------------------------------
_crop_rows = {}


def handwriting_crop(rng, script):
    """(image, text) of a short handwritten line (1-7 words) from the training writers, or None."""
    sources = [TRAIN[KHATT_TRAIN], TRAIN["memoir"]] if script == "ar" else [TRAIN["iam"], TRAIN["gnhk"]]
    sources = [ds for ds in sources if ds is not None and len(ds)]
    if not sources:
        return None
    ds = rng.choice(sources)
    if id(ds) not in _crop_rows:
        _crop_rows[id(ds)] = [i for i, t in enumerate(ds["text"]) if t and 1 <= len(t.split()) <= 7]
    rows = _crop_rows[id(ds)]
    if not rows:
        return None
    row = ds[rng.choice(rows)]
    return row["image"], re.sub(r"\s+", " ", row["text"]).strip()


def paste_crop(img, box, crop, right_aligned, colour):
    x0, y0, x1, y1 = (int(v) for v in box)
    a = np.asarray(crop.convert("L"), dtype=np.float32)
    lo, hi = np.percentile(a, 1), np.percentile(a, 90)
    if hi - lo > 30:
        a = np.clip((a - lo) / (hi - lo) * 255, 0, 255)
    a = a.astype(np.uint8)
    _, paper_mask = cv2.threshold(a, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    crop = Image.fromarray(np.where(paper_mask > 0, 255, a).astype(np.uint8))
    scale = min((y1 - y0 - 4) / crop.height, (x1 - x0 - 10) / crop.width, 1.5)
    if scale <= 0.05:
        return False
    crop = crop.resize((max(1, int(crop.width * scale)), max(1, int(crop.height * scale))), Image.LANCZOS)
    px = x1 - 5 - crop.width if right_aligned else x0 + 5
    py = y0 + (y1 - y0 - crop.height) // 2
    px, py = max(0, min(px, img.width - crop.width)), max(0, min(py, img.height - crop.height))
    region = img.crop((px, py, px + crop.width, py + crop.height))
    img.paste(Image.composite(Image.new("RGB", crop.size, colour), region, ImageOps.invert(crop)), (px, py))
    return True


LABEL_MODES = ["ar_right_en_left", "ar_right", "en_left", "stacked", "above", "table"]
VALUE_STYLES = ["underline", "dotted", "box", "comb", "plain"]


def pen_colour(rng, hand):
    return (rng.randint(10, 40), rng.randint(20, 60), rng.randint(110, 170)) if hand and rng.random() < 0.6 else tuple([rng.randint(0, 45)] * 3)


def value_rule(draw, style, region, rng, colour):
    x0, y0, x1, y1 = region
    if style == "underline":
        draw.line([(x0, y1), (x1, y1)], fill=colour, width=rng.choice([1, 2]))
    elif style == "dotted":
        for x in range(int(x0), int(x1), 7):
            draw.line([(x, y1), (min(x1, x + 2), y1)], fill=colour, width=2)
    elif style == "box":
        draw.rectangle((x0, y0, x1, y1), outline=colour, width=rng.choice([1, 2]))


def write_value(img, draw, region, shown, font_path, rng, hand, comb, rule_colour):
    """Draw a value in its region: fitted to the height and width, right-aligned for Arabic, one character per cell in a comb."""
    x0, y0, x1, y1 = region
    size = max(12, min(40, int((y1 - y0) * 0.66)))
    font = get_font(font_path, size)
    ink = pen_colour(rng, hand)
    if comb:                                       # digits and codes, one character per cell, read left to right
        cells = len(shown) + rng.randint(0, 3)
        cell = min((x1 - x0) / cells, (y1 - y0) * 0.9)
        start = x1 - cells * cell if rng.random() < 0.5 else x0
        small = get_font(font_path, max(10, int(cell * 0.7)))
        for c in range(cells):
            draw.rectangle((start + c * cell, y1 - cell, start + (c + 1) * cell, y1), outline=rule_colour, width=1)
        for c, ch in enumerate(shown):
            if ch.strip():
                draw_text(draw, start + c * cell, start + (c + 1) * cell, y1 - cell * 0.9, ch, small, ink, "center")
        return
    shown = typeset_ltr(shown)
    while text_width(draw, shown, font) > (x1 - x0) - 10 and size > 12:
        size -= 2
        font = get_font(font_path, size)
    width = text_width(draw, shown, font)
    shift = rng.uniform(0, 0.25) * max(0, (x1 - x0) - width - 10) if hand else 0      # handwriting starts a little in
    x = x1 - 5 - width - shift if arabic_font_needed(shown) else x0 + 5 + shift
    x = min(max(x, x0), max(x0, x1 - width))                                           # never outside its region
    y = y1 - size * 1.3 + (rng.uniform(-0.1, 0.1) * size if hand else 0)
    shown_text, options = prepare(shown)
    draw.text((x, y), shown_text, font=font, fill=ink, **options)


def form_v2_example(rng, damage_level=None, heldout=False):
    keys = rng.sample(sorted(FIELD_TYPES), rng.randint(4, 10))
    mode = rng.choice(LABEL_MODES)
    style = "plain" if mode == "table" else rng.choice(VALUE_STYLES)
    columns = 2 if (mode in ("stacked", "above", "ar_right", "en_left") and rng.random() < 0.35) else 1
    language = "en" if mode == "en_left" else rng.choice(["ar", "ar", "bilingual"])
    size = rng.randint(20, 30)
    label_ar, label_en = rng.choice(ARABIC_BODY_FONTS), rng.choice(ENGLISH_BODY_FONTS)
    typed = {"ar": rng.choice(ARABIC_BODY_FONTS), "en": rng.choice(ENGLISH_BODY_FONTS + MONO_FONTS)}
    p_hand, p_empty = rng.choice([0.0, 0.3, 0.6, 0.9]), 0.08
    sentences = {s: POOLS["msa" if s == "ar" else "en"][1 if heldout else 0] for s in ("ar", "en")}
    width, pad = rng.randint(1100 if columns == 2 else 820, 1300), rng.randint(28, 50)
    tall = mode in ("stacked", "above")
    row_h = int(size * (3.4 if tall else 2.3))
    gap = 30
    col_w = (width - 2 * pad - (columns - 1) * gap) / columns
    plan, sections = [], rng.random() < 0.4
    for i in range(0, len(keys), columns):
        if sections and (i == 0 or rng.random() < 0.3):
            plan.append(("section", rng.choice(VOCAB["sections"])))
        plan.append(("fields", keys[i:i + columns]))
    ticks = rng.random() < 0.3
    items = rng.random() < 0.2
    header_h, footer_h = int(size * 3.4), int(row_h * 2.2) if rng.random() < 0.6 else pad
    height = (2 * pad + header_h + sum(int(size * 1.9) if kind == "section" else row_h for kind, _ in plan)
              + (row_h if ticks else 0) + (5 * int(size * 1.6) if items else 0) + footer_h)
    img = Image.new("RGB", (width, height), tuple(rng.randint(238, 255) for _ in range(3)))
    draw = ImageDraw.Draw(img)
    grey = (rng.randint(30, 80),) * 3
    rule = (rng.randint(70, 150),) * 3
    if rng.random() < 0.3:                                                                  # page border
        draw.rectangle((pad // 3, pad // 3, width - pad // 3, height - pad // 3), outline=rule, width=2)
    title_ar, title_en = rng.choice(VOCAB["titles"])
    if rng.random() < 0.3:                                                                  # coloured title band
        draw.rectangle((pad, pad, width - pad, pad + header_h - size), fill=tuple(rng.randint(205, 235) for _ in range(3)))
    if rng.random() < 0.4:                                                                  # logo placeholder
        lx = pad + 6 if rng.random() < 0.5 else width - pad - size * 2 - 6
        draw.rectangle((lx, pad + 6, lx + size * 2, pad + 6 + size * 2), outline=(120, 120, 120), width=2)
    title_font = get_font(label_ar, size + 6)
    if rng.random() < 0.4:
        draw_text(draw, pad, width - pad, pad + 4, title_ar, title_font, (0, 0, 0), "center")
        draw_text(draw, pad, width - pad, pad + size * 1.9, title_en, get_font(label_en, int(size * 0.9)), (0, 0, 0), "center")
    else:
        draw_text(draw, pad, width - pad, pad + 4, title_ar, title_font, (0, 0, 0), "right")
        draw_text(draw, pad + size * 3, width - pad, pad + size // 2, title_en, get_font(label_en, int(size * 0.9)), (0, 0, 0), "left")
    y = pad + header_h
    if ticks:                                                                                # decorative tick boxes (not in the JSON)
        options = rng.choice(VOCAB["choices"])[:3]
        ticked, x = rng.randrange(len(options)), width - pad
        for i, (oa, oe) in enumerate(options):
            box = (x - size, y + 4, x, y + 4 + size)
            draw.rectangle(box, outline=grey, width=2)
            if i == ticked:
                draw.line([(box[0] + 4, box[1] + size // 2), (box[0] + size // 2, box[3] - 4), (box[2] - 3, box[1] + 3)], fill=(0, 0, 0), width=3)
            draw_text(draw, pad, x - size - 10, y, oa if language != "en" else oe, get_font(label_ar if language != "en" else label_en, size), grey, "right")
            x -= int(col_w / 3) if columns == 1 else size * 6
        y += row_h
    answer = {}
    for kind, content in plan:
        if kind == "section":
            sa, se = content
            if rng.random() < 0.5:
                draw.rectangle((pad, y + 2, width - pad, y + int(size * 1.6)), fill=tuple(rng.randint(215, 240) for _ in range(3)))
            draw_text(draw, pad + 6, width - pad - 6, y + 4, sa, get_font(label_ar, size), (0, 0, 0), "right")
            if language != "ar":
                draw_text(draw, pad + 6, width - pad - 6, y + 6, se, get_font(label_en, int(size * 0.85)), (0, 0, 0), "left")
            y += int(size * 1.9)
            continue
        for c, key in enumerate(content):
            x1 = width - pad - c * (col_w + gap)                                             # columns fill from the right
            x0 = x1 - col_w
            ar_label = rng.choice(FIELD_TYPES[key]["ar"]) + rng.choice([":", "", " :"])
            en_label = rng.choice(FIELD_TYPES[key]["en"]) + ":"
            fa, fe = get_font(label_ar, size), get_font(label_en, int(size * 0.8))
            top, line_y = y + 4, y + int(size * 1.6)
            if mode == "table":
                divider = x1 - col_w * 0.38
                draw.rectangle((x0, y, x1, y + row_h - 4), outline=rule, width=1)
                draw.line([(divider, y), (divider, y + row_h - 4)], fill=rule, width=1)
                draw_text(draw, divider + 6, x1 - 6, top, ar_label, fa, grey, "right")
                region = (x0 + 6, y + 4, divider - 6, y + row_h - 10)
            elif mode in ("stacked", "above"):
                if language == "en":
                    draw_text(draw, x0, x1, top, en_label, fe, grey, "left")
                elif mode == "above" and language == "bilingual":
                    draw_text(draw, x0, x1, top, f"{ar_label} / {en_label}", fa, grey, "right")
                else:
                    draw_text(draw, x0, x1, top, ar_label, fa, grey, "right")
                    if mode == "stacked" and language == "bilingual":
                        draw_text(draw, x0, x1, top + 2, en_label, fe, grey, "left")
                region = (x0 + 4, y + int(size * 1.5), x1 - 4, y + row_h - 10)
            else:
                right, left = x1, x0
                if mode in ("ar_right_en_left", "ar_right"):
                    draw_text(draw, x0, x1, top, ar_label, fa, grey, "right")
                    right = x1 - text_width(draw, ar_label, fa) - 20
                if mode in ("ar_right_en_left", "en_left"):
                    draw_text(draw, x0, x1, top + 2, en_label, fe, grey, "left")
                    left = x0 + text_width(draw, en_label, fe) + 20
                region = (min(left, right - 60), y, right, line_y)                      # never narrower than 60 px
            field_style = style
            kind_of = FIELD_KIND[key]
            script = "en" if (language == "en" or kind_of in ("name_en", "email")) else "ar" if language == "ar" else rng.choice(["ar", "ar", "en"])
            if rng.random() < p_empty:                                                      # an empty field: null
                value_rule(draw, "box" if field_style == "comb" else field_style, region, rng, rule)
                answer[key] = None
                continue
            hand = rng.random() < p_hand
            if kind_of == "choice":                                                         # tick one option
                options = rng.choice(VOCAB["choices"])[:3]
                ticked = rng.randrange(len(options))
                x = region[2]
                for i, option in enumerate(options):
                    text = option[0] if script == "ar" else option[1]
                    box = (x - size * 0.8, region[1] + 4, x, region[1] + 4 + size * 0.8)
                    draw.rectangle(box, outline=grey, width=2)
                    if i == ticked:
                        draw.line([(box[0] + 3, box[1] + size * 0.4), (box[0] + size * 0.35, box[3] - 3), (box[2] - 2, box[1] + 2)], fill=pen_colour(rng, hand), width=3)
                        answer[key] = text
                    draw_text(draw, region[0], x - size, region[1] + 2, text, get_font(label_ar if script == "ar" else label_en, int(size * 0.85)), grey, "right")
                    x -= max(size * 4, (region[2] - region[0]) / len(options))
                continue
            if hand and kind_of in ("free_text", "address") and rng.random() < 0.7:
                crop = handwriting_crop(rng, script)
                if crop is not None:
                    value_rule(draw, "box" if field_style == "comb" else field_style, region, rng, rule)
                    if paste_crop(img, region, crop[0], script == "ar", pen_colour(rng, True)):
                        answer[key] = crop[1]
                        continue
            value, shown = make_value(kind_of, rng, script, sentences[script])
            comb = field_style == "comb" and len(shown) <= 16 and not is_arabic(shown)
            if not comb:
                value_rule(draw, "box" if field_style == "comb" else field_style, region, rng, rule)
            script_font = "ar" if arabic_font_needed(shown) else "en"
            font = rng.choice(HAND_FONTS[script_font]) if hand else typed[script_font]
            write_value(img, draw, region, shown, font, rng, hand, comb, rule)
            answer[key] = value
        y += row_h
    if items:                                                                                 # a decorative item table
        cols = [0.0, 0.55, 0.75, 1.0]
        row = int(size * 1.6)
        for r in range(5):
            for a, b in zip(cols[:-1], cols[1:]):
                draw.rectangle((pad + (width - 2 * pad) * a, y + r * row, pad + (width - 2 * pad) * b, y + (r + 1) * row), outline=rule, width=1)
            cells = (["الوصف", "الكمية", "السعر"] if r == 0 else [text_piece(rng, POOLS["msa"][0], 3), str(rng.randint(1, 20)), f"{rng.randint(5, 900)}.00"])
            for (a, b), text in zip(zip(cols[:-1], cols[1:]), cells):
                draw_text(draw, pad + (width - 2 * pad) * a + 6, pad + (width - 2 * pad) * b - 6, y + r * row + 3, text,
                          get_font(label_ar, int(size * 0.8)), grey, "right")
        y += 5 * row
    if footer_h > pad:
        if rng.random() < 0.6:
            draw_stamp(draw, rng, (pad, y + 6, pad + int(row_h * 1.9), y + 6 + int(row_h * 1.9)), rng.choice(["مصدق", "مدفوع", "مستلم"]))
        if rng.random() < 0.6:
            draw_signature(draw, rng, (width // 2 - row_h * 2, y + 6, width // 2 + row_h * 2, y + row_h * 2))
        if rng.random() < 0.5:
            draw_barcode(draw, rng, (width - pad - row_h * 4, y + row_h // 2, width - pad, y + row_h * 2))
    order = list(answer)
    if rng.random() < 0.5:
        rng.shuffle(order)
    level = damage_level if damage_level is not None else rng.choice([0, 1, 1, 2])
    return damage(img, rng, level), form_instruction_v6(order), json.dumps({k: answer[k] for k in order}, ensure_ascii=False)


def template_form_example(rng):
    """A training template filled by part 1b, photographed at a slight angle half the time."""
    ds = TRAIN["forms_template"]
    row = ds[rng.randrange(len(ds))]
    img = row["image"].convert("RGB")
    if rng.random() < 0.5:
        img = perspective(img, rng, rng.uniform(0.01, 0.04))
    return damage(img, rng, rng.choice([0, 1, 1, 2])), form_instruction_v6(list(json.loads(row["json"]))), row["json"]


show_examples(form_v2_example, n=3)

## F. Models, reading and scoring

- `load_model(path)` loads our checkpoints (Qwen2.5-VL class) and, through `AutoModelForImageTextToText`, other
  models (Qwen3-VL-2B, AIN-7B). Image size limits are set in visual tokens, converted with each model's own patch
  size (14 px for Qwen2.5-VL, 16 px for Qwen3-VL). `adapter=` loads a LoRA adapter on top without merging it.
- `make_batch` and `read_images` are v5's (the loss covers only the answer; the token limit grows with the image).
  Decoding is set **per task**: text lines use `DEFAULTS["line_decoding"]` (phase 1 decides whether that is
  `no_repeat_ngram_size=4`), forms always decode without repetition control, because JSON legitimately repeats `", "`.
- **Schema-forced JSON** (`read_forms(..., schema=True)`): the keys are written by the code, not by the model. For each
  key the prompt is extended with `{"key1": value1, "key2":` and the model generates only the value, stopping as soon as
  it has written a complete JSON value (a string, a number or `null`). The answer then always parses and has exactly
  the requested keys. This is the effect of Outlines' schema decoding without its dependency.

**Text scores** (unchanged from v5, plus two): CER/WER raw, without diacritics, Arabic-normalised, answer length, and
per set where it applies: **letters-only** (AraMS-28k, whose labels keep only Arabic letters; both sides reduced the
same way) and **punctuation spacing** (IAM, whose labels put spaces before punctuation). Edit distances come from
`rapidfuzz` (fast C++), so pages of 400 words score quickly.

**Form scores** (new): strict JSON (the whole answer parses as one object), lenient JSON (v5's "a `{...}` span
parses"), exact match per field, split by how the value was written (printed, handwritten, empty), ANLS* (average
normalised Levenshtein similarity over the union of keys, so extra keys cost), whole-form exact match, and the mean
number of extra and missing keys per form.

**Visual dependence** (from v5's Results 6): per-image answer loss with the image's own picture, with another picture
of the same set resized to the same size, and with a blank page. Specificity = (mismatched − matched) ÷ (blank −
matched); own-image preference = share of images whose own picture gives the lower loss. The roadmap's guard: every
run keeps specificity above 1.

In [ ]:
from transformers import StoppingCriteria, StoppingCriteriaList


def model_type_of(path):
    config = os.path.join(path, "config.json")
    return json.load(open(config)).get("model_type") if os.path.exists(config) else AutoConfig.from_pretrained(path).model_type


def processor_for(path, max_tokens=None):
    """The processor with image size limits in visual tokens (converted with the model's own patch and merge size)."""
    image_processor = AutoImageProcessor.from_pretrained(path)
    unit = (getattr(image_processor, "patch_size", 14) * getattr(image_processor, "merge_size", 2)) ** 2
    return AutoProcessor.from_pretrained(path, size={"shortest_edge": MIN_TOKENS * unit, "longest_edge": (max_tokens or MAX_TOKENS) * unit})


def load_model(path, dtype=torch.float32, max_tokens=None, adapter=None):
    if model_type_of(path) == "qwen2_5_vl":
        cls = Qwen2_5_VLForConditionalGeneration
    else:
        from transformers import AutoModelForImageTextToText
        cls = AutoModelForImageTextToText
    model = cls.from_pretrained(path, dtype=dtype).to(DEVICE)
    if adapter:
        model = PeftModel.from_pretrained(model, adapter)
    processor = processor_for(path, max_tokens)
    print(f"loaded {path}{' + adapter ' + adapter if adapter else ''}: {count_parameters(model) / 1e9:.3f}B parameters")
    return model, processor


def count_parameters(model, trainable_only=False):
    return sum(p.numel() for p in model.parameters() if p.requires_grad or not trainable_only)


def free_memory():
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()


def chat(instruction, answer=None):
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": instruction}]}]
    if answer is not None:
        messages.append({"role": "assistant", "content": [{"type": "text", "text": answer}]})
    return messages


def make_batch(examples, processor):
    tok = processor.tokenizer
    tok.padding_side = "right"
    texts = [processor.apply_chat_template(chat(instruction, answer), tokenize=False) for _, instruction, answer in examples]
    batch = processor(text=texts, images=[img for img, _, _ in examples], padding=True, return_tensors="pt")
    labels = batch["input_ids"].clone()
    labels[batch["attention_mask"] == 0] = -100
    header = tok("<|im_start|>assistant\n", add_special_tokens=False).input_ids
    for row, ids in enumerate(batch["input_ids"].tolist()):
        starts = [j for j in range(len(ids) - len(header) + 1) if ids[j:j + len(header)] == header]
        labels[row, :starts[-1] + len(header)] = -100
    batch["labels"] = labels
    return batch


def amp():
    return torch.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=USE_AMP)


def sync():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()


TOKEN_LIMITS = {"line": (32, 1.0, 512), "page": (32, 1.0, 900), "form": (400, 0.0, 400)}   # kind -> (base, per visual token, ceiling)


def max_tokens_for(kind, visual_tokens):
    base, per_token, ceiling = TOKEN_LIMITS[kind]
    return 16 if QUICK_TEST else int(min(ceiling, base + per_token * visual_tokens))


def clean_output(text):
    return text.translate(OUTPUT_CHAR_MAP).strip()


def visual_tokens_of(inputs, processor):
    grid = inputs["image_grid_thw"]
    return int((grid[:, 0] * grid[:, 1] * grid[:, 2]).max()) // getattr(processor.image_processor, "merge_size", 2) ** 2


@torch.no_grad()
def read_images(model, processor, examples, kind="line", batch_size=8, decoding=None, max_new_tokens=None, timings=None):
    """Generate answers (greedy, plus the task's decoding settings). Left padding, so every prompt ends where the answer starts."""
    tok = processor.tokenizer
    tok.padding_side = "left"
    was_training = model.training
    model.eval()
    model.config.use_cache = True
    settings = (DEFAULTS["line_decoding"] if kind != "form" else {}) if decoding is None else decoding
    answers = []
    for i in range(0, len(examples), batch_size):
        part = examples[i:i + batch_size]
        texts = [processor.apply_chat_template(chat(instruction), tokenize=False, add_generation_prompt=True) for _, instruction, _ in part]
        inputs = processor(text=texts, images=[img for img, _, _ in part], padding=True, return_tensors="pt").to(DEVICE)
        limit = max_new_tokens or max_tokens_for(kind, visual_tokens_of(inputs, processor))
        sync()
        start = time.perf_counter()
        with amp():
            out = model.generate(**inputs, max_new_tokens=limit, do_sample=False, **settings)
        sync()
        new = out[:, inputs["input_ids"].shape[1]:]
        if timings is not None:
            timings.append((time.perf_counter() - start, int((new != tok.pad_token_id).sum())))
        answers += [clean_output(a) for a in processor.batch_decode(new, skip_special_tokens=True)]
    tok.padding_side = "right"
    model.config.use_cache = False
    model.train(was_training)
    return answers


# --- schema-forced JSON -----------------------------------------------------------------------------------------------
INCOMPLETE = object()


def json_value_at_start(text):
    """(value, end) for the complete JSON value at the start of `text` (after spaces), or (INCOMPLETE, None)."""
    s = text.lstrip()
    offset = len(text) - len(s)
    if s.startswith("null"):
        return None, offset + 4
    if "null".startswith(s) and s:
        return INCOMPLETE, None
    if s.startswith('"'):
        escaped = False
        for i in range(1, len(s)):
            c = s[i]
            if escaped:
                escaped = False
            elif c == "\\":
                escaped = True
            elif c == '"':
                try:
                    return json.loads(s[:i + 1]), offset + i + 1
                except Exception:
                    return s[1:i], offset + i + 1
        return INCOMPLETE, None
    m = re.match(r"[^,}\n]*[,}\n]", s)                     # a bare number or word, ended by , } or a new line
    if m:
        value = m.group(0)[:-1].strip()
        return (value or None), offset + m.end() - 1
    return INCOMPLETE, None


class ValueComplete(StoppingCriteria):
    """Stops each row once its newly generated text holds a complete JSON value."""

    def __init__(self, tokenizer, prompt_length):
        self.tokenizer, self.prompt_length = tokenizer, prompt_length

    def __call__(self, input_ids, scores, **kwargs):
        done = [json_value_at_start(self.tokenizer.decode(row[self.prompt_length:], skip_special_tokens=True))[0] is not INCOMPLETE
                for row in input_ids]
        return torch.tensor(done, dtype=torch.bool, device=input_ids.device)


@torch.no_grad()
def read_forms_schema(model, processor, examples, batch_size=4, max_value_tokens=64):
    tok = processor.tokenizer
    tok.padding_side = "left"
    was_training = model.training
    model.eval()
    model.config.use_cache = True
    outputs = []
    for i in range(0, len(examples), batch_size):
        part = examples[i:i + batch_size]
        keys = [list(json.loads(answer)) for _, _, answer in part]
        prompts = [processor.apply_chat_template(chat(instruction), tokenize=False, add_generation_prompt=True) for _, instruction, _ in part]
        values, written = [{} for _ in part], ["{" for _ in part]
        for j in range(max(len(k) for k in keys)):
            rows = [r for r in range(len(part)) if j < len(keys[r])]
            for r in rows:
                written[r] += ("" if j == 0 else ", ") + json.dumps(keys[r][j], ensure_ascii=False) + ":"
            inputs = processor(text=[prompts[r] + written[r] for r in rows], images=[part[r][0] for r in rows],
                               padding=True, return_tensors="pt").to(DEVICE)
            length = inputs["input_ids"].shape[1]
            with amp():
                out = model.generate(**inputs, max_new_tokens=max_value_tokens, do_sample=False,
                                     stopping_criteria=StoppingCriteriaList([ValueComplete(tok, length)]))
            for row, r in enumerate(rows):
                text = tok.decode(out[row, length:], skip_special_tokens=True)
                value, _ = json_value_at_start(text)
                if value is INCOMPLETE:                    # stopped by the token limit or end of text: keep what was written
                    value = text.strip().strip('",}').strip() or None
                value = value if value is None else clean_output(str(value))
                values[r][keys[r][j]] = value
                written[r] += " " + json.dumps(value, ensure_ascii=False)
        outputs += [json.dumps(v, ensure_ascii=False) for v in values]
    tok.padding_side = "right"
    model.config.use_cache = False
    model.train(was_training)
    return outputs


def read_forms(model, processor, examples, schema=None, batch_size=4):
    schema = DEFAULTS["form_schema"] if schema is None else schema
    if schema:
        return read_forms_schema(model, processor, examples, batch_size)
    return read_images(model, processor, examples, kind="form", batch_size=batch_size, decoding={})


# --- text scores --------------------------------------------------------------------------------------------------------
try:
    from rapidfuzz.distance import Levenshtein as _levenshtein

    def edit_distance(a, b):
        return _levenshtein.distance(a, b)
except ImportError:
    def edit_distance(a, b):
        previous = list(range(len(b) + 1))
        for i, ca in enumerate(a, 1):
            current = [i]
            for j, cb in enumerate(b, 1):
                current.append(min(previous[j] + 1, current[j - 1] + 1, previous[j - 1] + (ca != cb)))
            previous = current
        return previous[-1]

ARABIC_LETTER_MAP = str.maketrans({"أ": "ا", "إ": "ا", "آ": "ا", "ٱ": "ا", "ى": "ي", "ة": "ه"})
TASHKIL_ALL = re.compile(r"[ؐ-ًؚ-ٰٟۖ-ۜ۟-۪ۤۧۨ-ۭ]")


def normalise(text):
    return re.sub(r"\s+", " ", unicodedata.normalize("NFC", text or "")).strip()


def normalise_arabic(text):
    return normalise(strip_marks(text)).translate(ARABIC_LETTER_MAP)


def letters_only(text):
    """AraMS-28k's own target cleaning: no tashkeel, only Arabic letters and spaces."""
    text = re.sub(r"[^ء-غف-ي ]", "", TASHKIL_ALL.sub("", normalise(text)))
    return re.sub(r" +", " ", text).strip()


def punct_spacing(text):
    """IAM puts spaces before punctuation ("word , word ."): remove spaces next to punctuation, on both sides alike."""
    return re.sub(r"\s*([^\w\s])\s*", r"\1", normalise(text))


NORMALISERS = {"letters_only": letters_only, "punct": punct_spacing}


def error_rates(predictions, truths):
    chars = [(edit_distance(p, t), len(t)) for p, t in zip(predictions, truths)]
    words = [(edit_distance(p.split(), t.split()), len(t.split())) for p, t in zip(predictions, truths)]
    return (sum(e for e, _ in chars) / max(1, sum(n for _, n in chars)), sum(e for e, _ in words) / max(1, sum(n for _, n in words)), chars)


def bootstrap_interval(pairs, rounds=1000, seed=0):
    if len(pairs) < 2:
        return (float("nan"), float("nan"))
    rng = np.random.default_rng(seed)
    edits, lengths = np.array([e for e, _ in pairs], dtype="float64"), np.array([n for _, n in pairs], dtype="float64")
    samples = rng.integers(0, len(pairs), size=(rounds, len(pairs)))
    rates = edits[samples].sum(1) / np.maximum(1, lengths[samples].sum(1))
    return (float(np.percentile(rates, 2.5)), float(np.percentile(rates, 97.5)))


def text_scores(predictions, truths, interval=False, normaliser=None):
    """CER/WER as written, without diacritics, Arabic-normalised, and with the set's own normaliser if it has one.
    `main` is the number to quote for the set (letters-only for AraMS, raw otherwise)."""
    p, t = [normalise(x) for x in predictions], [normalise(x) for x in truths]
    cer, wer, pairs = error_rates(p, t)
    cer_plain, wer_plain, _ = error_rates([strip_marks(x) for x in p], [strip_marks(x) for x in t])
    cer_norm, wer_norm, _ = error_rates([normalise_arabic(x) for x in p], [normalise_arabic(x) for x in t])
    scores = {"cer": cer, "wer": wer, "length": sum(map(len, p)) / max(1, sum(map(len, t))), "cer_no_marks": cer_plain,
              "wer_no_marks": wer_plain, "cer_norm": cer_norm, "wer_norm": wer_norm, "images": len(p), "main": cer, "main_wer": wer}
    main_pairs = pairs
    if normaliser:
        f = NORMALISERS[normaliser]
        c, w, main_pairs_n = error_rates([f(x) for x in p], [f(x) for x in t])
        scores[f"cer_{normaliser}"], scores[f"wer_{normaliser}"] = c, w
        if normaliser == "letters_only":
            scores["main"], scores["main_wer"], main_pairs = c, w, main_pairs_n
    if interval:
        scores["cer_ci"] = bootstrap_interval(pairs)
        scores["main_ci"] = bootstrap_interval(main_pairs)
    return scores


# --- form scores --------------------------------------------------------------------------------------------------------
def parse_answer(text):
    """(object if the whole answer is one JSON object else None, object from the first {...} span else None)."""
    strict = lenient = None
    try:
        value = json.loads(text.strip())
        strict = value if isinstance(value, dict) else None
    except Exception:
        pass
    lenient = strict
    if lenient is None:
        m = re.search(r"\{.*\}", text, re.S)
        if m:
            try:
                value = json.loads(m.group(0))
                lenient = value if isinstance(value, dict) else None
            except Exception:
                pass
    return strict, lenient


def field_text(value):
    """A field value as compared: None and empty count as empty; Western digits; whitespace normalised."""
    if value is None:
        return None
    text = normalise(str(value)).translate(TO_WESTERN_DIGITS)
    return text or None


def anls(predicted, truth, tau=0.5):
    p, t = field_text(predicted), field_text(truth)
    if t is None or p is None:
        return 1.0 if p == t else 0.0
    d = edit_distance(p, t) / max(len(p), len(t))
    return 1.0 - d if d < tau else 0.0


def form_scores(predictions, examples, kinds=None):
    """kinds: for each form, {key: printed | handwritten | empty | unknown} (None: all unknown)."""
    n = len(examples)
    strict_ok = lenient_ok = whole = 0
    anls_total, extra_total, missing_total = 0.0, 0, 0
    by_kind = defaultdict(lambda: [0, 0])                           # kind -> [correct, total]
    for i, (prediction, (_, _, answer)) in enumerate(zip(predictions, examples)):
        truth = json.loads(answer)
        strict, got = parse_answer(prediction)
        strict_ok += strict is not None
        lenient_ok += got is not None
        got = got or {}
        form_kinds = (kinds[i] if kinds else None) or {}
        all_right = True
        for key, value in truth.items():
            right = field_text(got.get(key)) == field_text(value)
            kind = form_kinds.get(key, "unknown")
            for bucket in ("all", kind):
                by_kind[bucket][0] += right
                by_kind[bucket][1] += 1
            all_right &= right
        extra, missing = len(set(got) - set(truth)), len(set(truth) - set(got))
        extra_total, missing_total = extra_total + extra, missing_total + missing
        whole += all_right and extra == 0
        union = set(truth) | set(got)
        anls_total += sum(anls(got.get(k), truth.get(k)) for k in union) / max(1, len(union))
    scores = {"forms": n, "strict_json": strict_ok / max(1, n), "valid_json": lenient_ok / max(1, n),
              "field_exact": by_kind["all"][0] / max(1, by_kind["all"][1]), "anls_star": anls_total / max(1, n),
              "whole_form": whole / max(1, n), "extra_keys": extra_total / max(1, n), "missing_keys": missing_total / max(1, n),
              "fields": by_kind["all"][1]}
    for kind in ("printed", "handwritten", "empty"):
        if by_kind[kind][1]:
            scores[f"field_exact_{kind}"], scores[f"fields_{kind}"] = by_kind[kind][0] / by_kind[kind][1], by_kind[kind][1]
    scores["field_accuracy"] = scores["field_exact"]               # v5's name for the same number
    return scores


# --- visual dependence (v5 Results 6) -----------------------------------------------------------------------------------
def donors(examples):
    """For each example, another example of the same set with different text (fixed, so every model sees the same pairs)."""
    n, out = len(examples), []
    for i in range(n):
        out.append(next((j for j in ((i + k) % n for k in range(n // 2, n // 2 + n))
                         if j != i and normalise(examples[j][2]) != normalise(examples[i][2])), None))
    return out


@torch.no_grad()
def answer_losses(model, processor, examples, batch_size=4):
    was_training = model.training
    model.eval()
    losses = []
    for i in range(0, len(examples), batch_size):
        part = examples[i:i + batch_size]
        batch = {k: v.to(DEVICE) for k, v in make_batch(part, processor).items()}
        labels = batch.pop("labels")
        with amp():
            logits = model(**batch).logits
        for r in range(len(part)):
            keep = labels[r, 1:] != -100
            losses.append(torch.nn.functional.cross_entropy(logits[r, :-1][keep].float(), labels[r, 1:][keep]).item())
        del logits
    model.train(was_training)
    return np.array(losses)


def visual_dependence(model, processor, examples, batch_size=4, rounds=1000, seed=0):
    pair = donors(examples)
    keep = [i for i, d in enumerate(pair) if d is not None]
    if len(keep) < 2:
        return None
    used = [examples[i] for i in keep]
    matched = answer_losses(model, processor, used, batch_size)
    mismatched = answer_losses(model, processor, [(examples[pair[i]][0].resize(examples[i][0].size, Image.BICUBIC), examples[i][1], examples[i][2])
                                                  for i in keep], batch_size)
    blank = answer_losses(model, processor, [(Image.new("RGB", img.size, "white"), ins, ans) for img, ins, ans in used], batch_size)
    gap_m, gap_b = mismatched - matched, blank - matched
    idx = np.random.default_rng(seed).integers(0, len(matched), size=(rounds, len(matched)))
    gm, gb = gap_m[idx].mean(1), gap_b[idx].mean(1)
    spec = np.where(gb > 0.1, gm / np.maximum(gb, 1e-6), np.nan) if gap_b.mean() > 0.1 else np.full(rounds, np.nan)
    return {"images": int(len(matched)), "matched_loss": float(matched.mean()), "mismatched_loss": float(mismatched.mean()),
            "blank_loss": float(blank.mean()), "specificity": float(gap_m.mean() / gap_b.mean()) if gap_b.mean() > 0.1 else float("nan"),
            "specificity_ci": [float(np.nanpercentile(spec, 2.5)), float(np.nanpercentile(spec, 97.5))] if np.isfinite(spec).any() else [float("nan")] * 2,
            "own_image_preference": float((matched < mismatched).mean())}

## G. The frozen test suite: scoring and tables
`score_suite(path, label)` loads a model once and scores every set in `eval_suite.json`, plus six generated sets kept
from v5 as regression guards (drawn Arabic, drawn Arabic with diacritics, drawn English, bilingual lines, v5 forms,
and v2 forms, 48 images each). For each set it records the score on **all** of the set's images and on its
**headline** (text-disjoint) subset, the visual-dependence test on the headline lines, and every prediction (in
`v6_predictions/<label>.json` on Drive). A label already scored is not scored again (`rescore=True` forces it), so
the comparison tables can be printed in any later session without a GPU.

`suite_table(labels)` prints the comparison: the main score per set with its 95% interval, the form scores, the
visual dependence, and the seen-versus-unseen writer gap. `gates(label)` checks a run against the roadmap's gates.

In [ ]:
N_GENERATED = 3 if QUICK_TEST else 48
GENERATED = {   # name: (kind, example maker); the seeds are v5's evaluation seeds
    "Drawn Arabic (held-out text)": ("line", lambda i: text_line_example(random.Random(10_000 + i), "ar", 1, heldout=True)),
    "Drawn Arabic with diacritics": ("line", lambda i: text_line_example(random.Random(15_000 + i), "ar", 1, heldout=True, source="tashkeel")),
    "Drawn English (held-out text)": ("line", lambda i: text_line_example(random.Random(20_000 + i), "en", 1, heldout=True)),
    "Drawn bilingual": ("line", lambda i: bilingual_line_example(random.Random(25_000 + i), heldout=True)),
    "Generated forms (v5 generator)": ("form", lambda i: form_example(random.Random(30_000 + i))),
    "Generated forms (v2 generator)": ("form", lambda i: form_v2_example(random.Random(35_000 + i), damage_level=1, heldout=True)),
}
SUITE_ENTRIES = [dict(e) for e in SUITE["sets"]] + [
    {"name": name, "kind": kind, "question": "generated (regression guard)", "normaliser": None, "tags": ["generated"], "generated": True,
     "indices": list(range(N_GENERATED)), "headline": list(range(N_GENERATED))}
    for name, (kind, _) in GENERATED.items()]
ENTRY = {e["name"]: e for e in SUITE_ENTRIES}
QUESTIONS = ["unseen Arabic handwriting", "historical handwriting", "printed Arabic", "English", "forms", "generated (regression guard)"]
VD_SETS = [e["name"] for e in SUITE_ENTRIES if e["kind"] == "line" and not e.get("generated")] + \
          ["Drawn Arabic (held-out text)", "Held-out forms (digital)", "Generated forms (v2 generator)"]
_generated_cache = {}


def entry_examples(entry):
    """(examples, rows, headline positions, kinds) for one suite set."""
    if entry.get("generated"):
        if entry["name"] not in _generated_cache:
            _generated_cache[entry["name"]] = [GENERATED[entry["name"]][1](i) for i in range(N_GENERATED)]
        examples = _generated_cache[entry["name"]]
        return examples, list(range(len(examples))), list(range(len(examples))), None
    ds = dataset(entry["dataset"])
    rows = entry["indices"][:SUITE_N] if SUITE_N else entry["indices"]
    headline = set(entry["headline"])
    positions = [k for k, r in enumerate(rows) if r in headline]
    examples, kinds = [], []
    for r in rows:
        row = ds[r]
        if entry["kind"] == "form":
            examples.append((row["image"].convert("RGB"), form_instruction_v6(list(json.loads(row["json"]))), row["json"]))
            kinds.append(json.loads(row["kinds"]))
        else:
            examples.append((row["image"].convert("RGB"), OCR_INSTRUCTION, row["text"]))
    return examples, rows, positions, (kinds if entry["kind"] == "form" else None)


def predictions_path(label):
    return os.path.join(PREDICTIONS_DIR, re.sub(r"[^\w\-]+", "_", label) + ".json")


def score_suite(path, label, sets=None, decoding=None, schema=None, clahe_sets=None, adapter=None, max_tokens=None,
                visual=True, rescore=False, model=None, processor=None, note=""):
    """Score one model on the suite (or the named sets). Pass model and processor to score a model already in memory."""
    _, results = results_file()
    done = results.get("scores", {}).get(label, {}).get("sets", {})
    names = [e["name"] for e in SUITE_ENTRIES if sets is None or e["name"] in sets]
    todo = names if rescore else [n for n in names if n not in done]
    if not todo:
        print(f"{label}: already scored on {len(names)} sets (rescore=True to score again)")
        return done
    decoding = DEFAULTS["line_decoding"] if decoding is None else decoding
    schema = DEFAULTS["form_schema"] if schema is None else schema
    clahe_sets = DEFAULTS["clahe_sets"] if clahe_sets is None else clahe_sets
    config = {"path": path, "adapter": adapter, "line_decoding": decoding, "form_schema": schema, "clahe_sets": clahe_sets,
              "max_tokens": max_tokens or MAX_TOKENS, "note": note, "date": time.strftime("%Y-%m-%d %H:%M")}
    own = model is None
    if own:
        free_memory()
        model, processor = load_model(path, dtype=AMP_DTYPE if DEVICE.type == "cuda" else torch.float32, max_tokens=max_tokens, adapter=adapter)
    saved = json.load(open(predictions_path(label))) if os.path.exists(predictions_path(label)) else {}
    started = time.time()
    for name in todo:
        entry, t = ENTRY[name], time.time()
        examples, rows, positions, kinds = entry_examples(entry)
        if not examples:
            continue
        if name in clahe_sets:
            examples = [(clahe(img), ins, ans) for img, ins, ans in examples]
        record = {"images": len(examples), "headline_images": len(positions)}
        if entry["kind"] == "form":
            predictions = read_forms(model, processor, examples, schema=schema)
            record["all"] = form_scores(predictions, examples, kinds)
            record["headline"] = record["all"]
            line = (f"strict JSON {record['all']['strict_json']:.2f}, fields {record['all']['field_exact']:.3f}, "
                    f"ANLS* {record['all']['anls_star']:.3f}, whole form {record['all']['whole_form']:.2f}")
        else:
            predictions = read_images(model, processor, examples, kind=entry["kind"], batch_size=8 if entry["kind"] == "line" else 2, decoding=decoding)
            truths = [answer for _, _, answer in examples]
            record["all"] = text_scores(predictions, truths, interval=True, normaliser=entry.get("normaliser"))
            record["headline"] = (record["all"] if len(positions) == len(examples) else
                                  text_scores([predictions[k] for k in positions], [truths[k] for k in positions], interval=True,
                                              normaliser=entry.get("normaliser")) if positions else None)
            h = record["headline"] or record["all"]
            line = f"CER {h['main']:.3f} [{h['main_ci'][0]:.3f}, {h['main_ci'][1]:.3f}] WER {h['main_wer']:.3f} length {h['length']:.2f}"
        if visual and name in VD_SETS:
            chosen = positions[:VD_MAX] if entry["kind"] != "form" else list(range(min(VD_MAX, len(examples))))
            record["vd"] = visual_dependence(model, processor, [examples[k] for k in chosen], batch_size=2 if entry["kind"] == "form" else 4)
            if record["vd"]:
                line += f" | specificity {record['vd']['specificity']:.2f}, own image {record['vd']['own_image_preference']:.0%}"
        saved[name] = {"rows": rows, "predictions": predictions}
        json.dump(saved, open(predictions_path(label), "w"), ensure_ascii=False)
        _, results = results_file()
        results.setdefault("scores", {}).setdefault(label, {"config": config, "sets": {}})
        results["scores"][label]["config"] = config
        results["scores"][label]["sets"][name] = record
        save_results(scores=results["scores"])
        print(f"  {label[:26]:26s} {name[:34]:34s} n={len(positions):4d}/{len(examples):<4d} {line}  ({(time.time() - t) / 60:.1f} min)")
    print(f"{label}: {len(todo)} sets in {(time.time() - started) / 60:.1f} min")
    if own:
        del model
        free_memory()
    return results_file()[1]["scores"][label]["sets"]


def scores_of(label):
    return results_file()[1].get("scores", {}).get(label, {}).get("sets", {})


def main_of(record, use="headline"):
    r = record.get(use) or record.get("all")
    return r.get("main", r.get("field_exact")) if r else None


def suite_table(labels, questions=None):
    """Main score per set (headline subset; CER, or field exact match for forms) for each label, with 95% intervals."""
    rows = {label: scores_of(label) for label in labels}
    width = 24
    print(f"{'set':36s}{'n':>5s}" + "".join(f"{l[:width - 2]:>{width}s}" for l in labels))
    for q in questions or QUESTIONS:
        names = [e["name"] for e in SUITE_ENTRIES if e["question"] == q and any(e["name"] in r for r in rows.values())]
        if not names:
            continue
        print(f"-- {q} " + "-" * 40)
        for name in names:
            first = next(r[name] for r in rows.values() if name in r)
            cells = []
            for label in labels:
                rec = rows[label].get(name)
                if rec is None:
                    cells.append("")
                elif ENTRY[name]["kind"] == "form":
                    cells.append(f"{rec['all']['field_exact']:.3f} / {rec['all']['strict_json']:.2f}")
                else:
                    h = rec.get("headline") or rec["all"]
                    cells.append(f"{h['main']:.3f} [{h['main_ci'][0]:.2f}-{h['main_ci'][1]:.2f}]")
            print(f"{name[:35]:36s}{first['headline_images']:5d}" + "".join(f"{c:>{width}s}" for c in cells))
    print("cells: CER [95% interval] on the headline (text-disjoint) subset; forms: field exact match / strict JSON rate")
    vd = [n for n in VD_SETS if any(rows[l].get(n, {}).get("vd") for l in labels)]
    if vd:
        print(f"\nvisual dependence: specificity (own-image preference); the guard is specificity above 1")
        for name in vd:
            print(f"{name[:35]:36s}     " + "".join(
                (f"{rows[l][name]['vd']['specificity']:.2f} ({rows[l][name]['vd']['own_image_preference']:.0%})" if rows[l].get(name, {}).get("vd") else "").rjust(width)
                for l in labels))
    unseen, seen = "KHATT unique-text test (official)", "KHATT seen-writer test (official)"
    if any(unseen in rows[l] and seen in rows[l] for l in labels):
        print(f"\nunseen-writer minus seen-writer CER (headline subsets):")
        print(f"{'':36s}     " + "".join((f"{main_of(rows[l][unseen]) - main_of(rows[l][seen]):+.3f}" if unseen in rows[l] and seen in rows[l] else "").rjust(width) for l in labels))


def form_table(labels, sets=None):
    sets = sets or [e["name"] for e in SUITE_ENTRIES if e["kind"] == "form"]
    measures = [("strict_json", "strict JSON"), ("field_exact", "field exact"), ("field_exact_printed", "printed"),
                ("field_exact_handwritten", "handwritten"), ("field_exact_empty", "empty (null)"), ("anls_star", "ANLS*"),
                ("whole_form", "whole form"), ("extra_keys", "extra keys"), ("missing_keys", "missing keys")]
    for name in sets:
        rows = [(l, scores_of(l).get(name)) for l in labels]
        rows = [(l, r) for l, r in rows if r]
        if not rows:
            continue
        print(f"\n{name} ({rows[0][1]['images']} forms)")
        print(f"{'':16s}" + "".join(f"{l[:22]:>24s}" for l, _ in rows))
        for key, title in measures:
            if any(key in r["all"] for _, r in rows):
                print(f"{title:16s}" + "".join((f"{r['all'][key]:.3f}" if key in r["all"] else "").rjust(24) for _, r in rows))


def gates(label, baseline="v5 stage 3"):
    """The roadmap's gates, checked on one label against the phase-0 baseline."""
    s, b = scores_of(label), scores_of(baseline)
    def m(name, use="headline"):
        return main_of(s[name], use) if name in s else None
    def ci_hi(rec):
        h = rec.get("headline") or rec["all"]
        return h["main_ci"][1]
    checks = [
        ("Gate 2: KHATT unique-text test (unseen writers) <= 0.15", m("KHATT unique-text test (official)"), lambda v: v <= 0.15),
        ("Gate 2: KITAB KHATT clean lines <= 0.25", m("KITAB KHATT"), lambda v: v <= 0.25),
        ("Gate 2: KITAB OnlineKhatt clean lines <= 0.15", m("KITAB OnlineKhatt"), lambda v: v <= 0.15),
        ("Gate 3: HistoryAr <= 0.40", m("HistoryAr"), lambda v: v <= 0.40),
        ("Gate 3: Muharaf test <= 0.12", m("Muharaf test"), lambda v: v <= 0.12),
        ("Gate 4: drawn English <= 0.005", m("Drawn English (held-out text)"), lambda v: v <= 0.005),
        ("Gate 5: held-out photos, strict JSON = 1.0", s.get("Held-out forms (photos)", {}).get("all", {}).get("strict_json"), lambda v: v >= 0.999),
        ("Gate 5: held-out photos, printed fields >= 0.90", s.get("Held-out forms (photos)", {}).get("all", {}).get("field_exact_printed"), lambda v: v >= 0.90),
        ("Gate 5: generated forms (v2) >= 0.98", s.get("Generated forms (v2 generator)", {}).get("all", {}).get("field_exact"), lambda v: v >= 0.98),
    ]
    print(f"gates for {label}:")
    for title, value, test in checks:
        if value is None:
            print(f"    --   {title}: not scored")
        else:
            print(f"   {'PASS' if test(value) else 'fail'}  {title}: {value:.3f}")
    for name in ["PATS", "SythenAR", "ISI-PPT", "EvArEST"]:
        if name in s and name in b:
            ok = m(name) <= ci_hi(b[name])
            print(f"   {'PASS' if ok else 'fail'}  printed Arabic stays inside {baseline}'s interval, {name}: {m(name):.3f} (baseline upper {ci_hi(b[name]):.3f})")
    low = [n for n in VD_SETS if s.get(n, {}).get("vd") and not (s[n]["vd"]["specificity"] > 1)]
    print(f"   {'PASS' if not low else 'fail'}  specificity above 1 on every set" + (f" (below on: {', '.join(low)})" if low else ""))

## H. Training: three methods, progress checks, saving, merging

`make_trainable(model, method)`:

| method | What trains | Learning rates | Optimizer |
| --- | --- | --- | --- |
| `"lora"` (v5's recipe) | LoRA rank 32 on the first 24 vision blocks and on the decoder; the last 8 vision blocks and the projector in full | adapters and projector at the run's `lr`; full blocks at 2e-5 | AdamW |
| `"full"` (phase 6) | every weight, except the first `FREEZE_VISION_BLOCKS` vision blocks if set | decoder 1e-4, projector 1e-4, vision 1e-5 (`FULL_LR`) | paged 8-bit AdamW |
| `"adapter"` (phase 5) | LoRA rank 32 on every vision block and the decoder; nothing else | the run's `lr` | AdamW |

`train(...)` is v5's loop (mixed precision, gradient checkpointing, warm-up then cosine decay to a tenth, skipped
non-finite steps, resume from the last check) with four additions: progress checks use **validation sets only**;
`select_on` keeps the weights of the best check on those sets (early stopping with `patience` checks without
improvement); `snapshot_at` saves the trainable weights at given fractions of the run, for the phase 6 checkpoint
merge; `max_steps` turns a run into a short pilot. Full fine-tuning saves about 7 GB of progress per check, so it
checks less often.

`save_checkpoint` folds LoRA adapters into the weights (adapters for forms are kept separate), saves in bf16 (2.3 GB),
reloads the saved copy and checks it gives the same output. `blend(model, other, alpha)` averages two checkpoints of
the same architecture weight by weight. `run_experiment(...)` does all of it: train, save, score on the suite, record.

In [ ]:
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

DECODER_LAYERS = r".*language_model\.layers\.\d+\.(self_attn\.(q_proj|k_proj|v_proj|o_proj)|mlp\.(gate_proj|up_proj|down_proj))$"


def base_of(model):
    return model.base_model.model if hasattr(model, "peft_config") else model


def vision_blocks(model):
    return base_of(model).model.visual.blocks


def vision_layers(model, skip_last):
    """Regex for the vision layers that get LoRA adapters: every block except the last `skip_last`."""
    depth = len(vision_blocks(model))
    ids = "|".join(str(i) for i in range(max(0, depth - skip_last)))
    return rf".*visual\.blocks\.({ids})\.(attn\.(qkv|proj)|mlp\.(gate_proj|up_proj|down_proj|linear_fc1|linear_fc2))$"


def report_trainable(model):
    n, total = count_parameters(model, trainable_only=True), count_parameters(model)
    print(f"trainable: {n / 1e6:,.1f}M of {total / 1e6:,.0f}M parameters ({100 * n / total:.2f}%)")


def make_trainable(model, method, decoder=True):
    for p in model.parameters():
        p.requires_grad = False
    if method == "full":
        for p in model.parameters():
            p.requires_grad = True
        if FREEZE_VISION_BLOCKS:
            for block in list(vision_blocks(model))[:FREEZE_VISION_BLOCKS]:
                for p in block.parameters():
                    p.requires_grad = False
            for p in base_of(model).model.visual.patch_embed.parameters():
                p.requires_grad = False
    else:
        full_blocks = VISION_FULL_BLOCKS if method == "lora" else 0
        targets = vision_layers(model, full_blocks)
        if decoder:
            targets = f"({targets})|({DECODER_LAYERS})"
        model = get_peft_model(model, LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.05, bias="none", target_modules=targets,
                                                 modules_to_save=["merger"] if method == "lora" else None))
        if full_blocks:
            blocks = vision_blocks(model)
            for block in list(blocks)[len(blocks) - full_blocks:]:
                for p in block.parameters():
                    p.requires_grad = True
    report_trainable(model)
    return model


def part_of(name):
    if "merger" in name:
        return "projector"
    if ".visual." in name or name.startswith("visual."):
        return "vision"
    return "decoder"


def param_groups(model, method, lr, lrs=None):
    named = [(n, p) for n, p in model.named_parameters() if p.requires_grad]
    if method == "full":
        lrs = {**FULL_LR, **(lrs or {})}
        groups = defaultdict(list)
        for n, p in named:
            groups[part_of(n)].append(p)
        return [{"params": ps, "lr": lrs[k], "base_lr": lrs[k], "name": k} for k, ps in groups.items()]
    adapters = [p for n, p in named if "lora_" in n or "modules_to_save" in n or "merger" in n]
    full = [p for n, p in named if not ("lora_" in n or "modules_to_save" in n or "merger" in n)]
    groups = [{"params": adapters, "lr": lr, "base_lr": lr, "name": "adapters and projector"}]
    if full:
        groups.append({"params": full, "lr": LR_FULL_BLOCKS, "base_lr": LR_FULL_BLOCKS, "name": "vision blocks in full"})
    return groups


def make_optimizer(groups, method):
    if method == "full" and OPTIMIZER_8BIT and DEVICE.type == "cuda":
        try:
            import bitsandbytes as bnb
            return bnb.optim.PagedAdamW8bit(groups, betas=(0.9, 0.95), weight_decay=0.0)
        except Exception as e:
            print(f"8-bit AdamW not available ({type(e).__name__}: {e}); using torch AdamW (more memory)")
    return torch.optim.AdamW(groups, betas=(0.9, 0.95), weight_decay=0.0)


# --- validation sets for the progress checks (never the test suite) -----------------------------------------------------
_check_sets = {}


def check_sets():
    """name -> (examples, kind, normaliser). Built once per session, from fixed seeds and fixed rows."""
    if _check_sets:
        return _check_sets
    n = N_CHECK
    _check_sets["drawn Arabic"] = ([text_line_example(random.Random(60_000 + i), "ar", 1, heldout=True) for i in range(n)], "line", None)
    _check_sets["drawn English"] = ([text_line_example(random.Random(61_000 + i), "en", 1, heldout=True) for i in range(n)], "line", None)
    _check_sets["handwriting style"] = ([handwriting_style_example(random.Random(64_000 + i), heldout=True) for i in range(n)], "line", None)
    for key, title, normaliser in [("khatt", "KHATT val", None), ("muharaf", "Muharaf val", None), ("memoir", "memoir val", None),
                                   ("arams", "AraMS val", "letters_only"), ("iam", "IAM val", "punct"), ("gnhk", "GNHK val", None),
                                   ("synthdog", "SynthDoG val", None)]:
        ds = VAL.get(key)
        if ds is None or len(ds) == 0:
            continue
        rows = sorted(random.Random(7).sample(range(len(ds)), min(n if key != "synthdog" else max(2, n // 2), len(ds))))
        _check_sets[title] = ([(ds[r]["image"].convert("RGB"), OCR_INSTRUCTION, ds[r]["text"]) for r in rows],
                              "page" if key == "synthdog" else "line", normaliser)
    _check_sets["forms v2"] = ([form_v2_example(random.Random(62_000 + i), damage_level=1, heldout=True) for i in range(n)], "form", None)
    _check_sets["forms v5"] = ([form_example(random.Random(63_000 + i)) for i in range(n)], "form", None)
    if VAL.get("forms_template") is not None:
        ds = VAL["forms_template"]
        _check_sets["template forms"] = ([(ds[r]["image"].convert("RGB"), form_instruction_v6(list(json.loads(ds[r]["json"]))), ds[r]["json"])
                                          for r in range(min(n, len(ds)))], "form", None)
    return _check_sets


def quick_check(model, processor, names):
    """Losses (own, swapped and blank image) on the drawn Arabic check set, and the main score of each named check set."""
    sets = check_sets()
    vd = visual_dependence(model, processor, sets["drawn Arabic"][0][:max(4, N_CHECK // 2)], rounds=200)
    scores = {"eval_loss": vd["matched_loss"], "mismatched_loss": vd["mismatched_loss"], "no_image_loss": vd["blank_loss"]} if vd else {}
    for name in names:
        if name not in sets:
            continue
        examples, kind, normaliser = sets[name]
        if kind == "form":
            scores[name] = form_scores(read_forms(model, processor, examples, schema=False), examples)["field_exact"]
        else:
            predictions = read_images(model, processor, examples, kind=kind, batch_size=8 if kind == "line" else 2)
            scores[name] = text_scores(predictions, [a for _, _, a in examples], normaliser=normaliser)["main"]
    return scores


def progress_path(name, suffix="in_progress"):
    return os.path.join(BASE_DIR, f"{RUN_NAME}_{name}_{suffix}.pt")


def train(model, processor, recipe, name, examples, method, micro_batch=4, accum=2, lr=2e-4, lrs=None, check_names=(),
          select_on=(), patience=None, snapshot_at=(), max_steps=None, checks=None, keep_files=True):
    """Train `model` on `examples` examples of `recipe`. check_names: validation sets scored at each check; select_on: the
    subset whose mean decides the best check (lower is better; form sets count as 1 - field accuracy)."""
    if QUICK_TEST:
        micro_batch, accum = 2, 1
    per_step = micro_batch * accum
    steps = max_steps or max(1, examples // per_step)
    checks = checks or (max(4, CHECKS_PER_RUN // 2) if method == "full" else CHECKS_PER_RUN)
    check_every = max(1, steps // checks)
    missing = [k for k in select_on if k not in check_sets()]
    if missing:   # e.g. no memoir lines (no HF_TOKEN) or no template forms yet: choose the best check on the sets that exist
        select_on = tuple(k for k in select_on if k in check_sets())
        print(f"  {', '.join(missing)}: not prepared, so the best check is chosen on {', '.join(select_on) or 'nothing (the last check is kept)'}")
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.enable_input_require_grads()
    model.config.use_cache = False
    model.train()
    params = {n: p for n, p in model.named_parameters() if p.requires_grad}
    optimizer = make_optimizer(param_groups(model, method, lr, lrs), method)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP and AMP_DTYPE == torch.float16)
    warmup = max(1, steps // 20)
    schedule = lambda step: (step + 1) / warmup if step < warmup else 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * (step - warmup) / max(1, steps - warmup)))
    names = list(dict.fromkeys(list(check_names) + list(select_on)))
    state = {"step": 0, "history": [], "skipped": 0, "minutes": 0.0, "best": None, "best_step": None, "bad_checks": 0, "stopped_early": False}
    if keep_files and os.path.exists(progress_path(name)):
        saved = torch.load(progress_path(name), map_location="cpu", weights_only=False)
        result = model.load_state_dict(saved["weights"], strict=False)
        assert not result.unexpected_keys, f"saved progress does not match this model: {result.unexpected_keys[:3]}"
        optimizer.load_state_dict(saved["optimizer"])
        scaler.load_state_dict(saved["scaler"])
        state.update(saved["state"])
        print(f"resuming {name} at step {state['step']}/{steps} ({state['minutes']:.0f} min of training so far)")

    def save_progress():
        if not keep_files:
            return
        tmp = progress_path(name) + ".tmp"
        torch.save({"weights": {n: p.detach().cpu() for n, p in params.items()}, "optimizer": optimizer.state_dict(),
                    "scaler": scaler.state_dict(), "state": state}, tmp)
        os.replace(tmp, progress_path(name))

    def check(step, train_loss):
        scores = quick_check(model, processor, names) if names else {}
        state["history"].append({"step": step, "examples": step * per_step, "train_loss": train_loss, **scores})
        line = f"  check @ step {step}: " + " | ".join(f"{k} {v:.3f}" for k, v in scores.items())
        if select_on and all(k in scores for k in select_on):
            value = float(np.mean([1 - scores[k] if check_sets()[k][1] == "form" else scores[k] for k in select_on]))
            if state["best"] is None or value < state["best"] - 1e-4:
                state["best"], state["best_step"], state["bad_checks"] = value, step, 0
                if keep_files:
                    torch.save({n: p.detach().cpu() for n, p in params.items()}, progress_path(name, "best"))
                line += f"  <- best so far ({value:.3f})"
            else:
                state["bad_checks"] += 1
        print(line)
        save_progress()

    print(f"{name}: {steps:,} steps x {per_step} examples = {steps * per_step:,} examples ({method}), check every {check_every} steps")
    if state["step"] == 0 and names:
        check(0, None)
    start_step = state["step"]
    batches = iter(DataLoader(GeneratedData(recipe, (steps - start_step) * per_step, seed=int(hashlib.md5(name.encode()).hexdigest()[:6], 16),
                                            offset=start_step * per_step),
                              batch_size=micro_batch, collate_fn=lambda ex: make_batch(ex, processor), num_workers=0 if QUICK_TEST else 2))
    started = run_started = time.time()
    recent, since_check = [], []
    snapshots = {max(1, int(f * steps)): f for f in snapshot_at}
    for step in range(start_step, steps):
        optimizer.zero_grad(set_to_none=True)
        step_loss, finite = 0.0, True
        for _ in range(accum):
            batch = {k: v.to(DEVICE) for k, v in next(batches).items()}
            with amp():
                loss = model(**batch).loss / accum
            if not torch.isfinite(loss):
                finite = False
                break
            scaler.scale(loss).backward()
            step_loss += loss.item()
        if not finite:
            state["skipped"] += 1
            if state["skipped"] > max(5, steps // 10):
                raise RuntimeError("Too many non-finite losses: lower the learning rate, or check the data.")
            continue
        for group in optimizer.param_groups:
            group["lr"] = group["base_lr"] * schedule(step)
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(list(params.values()), 1.0)
        scaler.step(optimizer)
        scaler.update()
        recent.append(step_loss)
        since_check.append(step_loss)
        state["step"] = step + 1
        minutes = state["minutes"] + (time.time() - started) / 60
        if (step + 1) % (1 if QUICK_TEST else 50) == 0 or step + 1 == steps:
            done_steps = step + 1 - start_step
            memory = f" | GPU peak {torch.cuda.max_memory_allocated() / 2**30:.1f} GB" if DEVICE.type == "cuda" else ""
            print(f"step {step + 1}/{steps} | loss {sum(recent) / len(recent):.4f} | lr x{schedule(step):.2f} | {minutes:.1f} min, "
                  f"about {(time.time() - run_started) / 60 / done_steps * (steps - step - 1):.0f} min left{memory}")
            recent = []
        if step + 1 in snapshots and keep_files:
            torch.save({n: p.detach().cpu() for n, p in params.items()}, progress_path(name, f"snap{int(100 * snapshots[step + 1])}"))
            print(f"  snapshot at {snapshots[step + 1]:.0%} of the run saved")
        if (step + 1) % check_every == 0 or step + 1 == steps:
            state["minutes"] = state["minutes"] + (time.time() - started) / 60
            started = time.time()
            check(step + 1, sum(since_check) / max(1, len(since_check)))
            since_check = []
            if patience and state["bad_checks"] >= patience:
                state["stopped_early"] = True
                print(f"  stopping early: no improvement on {', '.join(select_on)} for {patience} checks")
                break
    if select_on and state["best_step"] is not None and state["best_step"] != state["step"] and os.path.exists(progress_path(name, "best")):
        model.load_state_dict(torch.load(progress_path(name, "best"), map_location="cpu"), strict=False)
        print(f"  kept the weights of the best check (step {state['best_step']})")
        if state["best_step"] == 0:
            print(f"  note: no check beat the starting weights on {', '.join(select_on)}, so the saved model equals its base. "
                  "Look at the check lines above before running the next step.")
    state["minutes"] += (time.time() - started) / 60
    print(f"{name}: training done in {state['minutes']:.1f} min ({state['skipped']} steps skipped)")
    return {"steps": state["step"], "examples": state["step"] * per_step, "minutes": round(state["minutes"], 1), "skipped_steps": state["skipped"],
            "best_step": state["best_step"], "stopped_early": state["stopped_early"], "history": state["history"],
            "peak_gpu_gb": torch.cuda.max_memory_allocated() / 2**30 if DEVICE.type == "cuda" else 0.0}


def plot_history(history, title, save_as=None):
    keys = [k for k in history[-1] if k not in ("step", "examples", "train_loss")] if history else []
    if not keys:
        return
    fig, (left, right) = plt.subplots(1, 2, figsize=(13, 4))
    steps = [h["step"] for h in history]
    trained = [(h["step"], h["train_loss"]) for h in history if h.get("train_loss") is not None]
    if trained:
        left.plot(*zip(*trained), marker="o", label="training loss")
    for k, style in [("eval_loss", "-"), ("mismatched_loss", ":"), ("no_image_loss", "--")]:
        if k in history[-1]:
            left.plot(steps, [h.get(k) for h in history], style, marker="o", label=k.replace("_", " "))
    left.set_xlabel("step"), left.set_title("loss"), left.legend(fontsize=8)
    for k in keys:
        if not k.endswith("loss"):
            right.plot(steps, [h.get(k) for h in history], marker="o", label=k)
    right.set_xlabel("step"), right.set_title("check sets (CER; forms: field accuracy)"), right.legend(fontsize=7)
    fig.suptitle(title)
    fig.tight_layout()
    if save_as:
        fig.savefig(save_as, dpi=130)
    plt.show()


def save_checkpoint(model, processor, out, adapter=False):
    """Save a trained model (LoRA folded in) or only its adapter. Returns the model as saved."""
    os.makedirs(out, exist_ok=True)
    if adapter:
        model.save_pretrained(out)
        processor.save_pretrained(out)
        print(f"adapter saved to {out}")
        return model
    if hasattr(model, "merge_and_unload"):
        model = model.merge_and_unload()
    model.config.use_cache = True
    if SAVE_BF16:
        model = model.to(torch.bfloat16)
    model.eval()
    probe = {k: v.to(DEVICE) for k, v in make_batch([check_sets()["drawn Arabic"][0][0]], processor).items()}
    with torch.no_grad(), amp():
        before = model(**probe).logits.float()
    model.save_pretrained(out)
    processor.save_pretrained(out)
    reloaded = type(model).from_pretrained(out, dtype=next(model.parameters()).dtype).to(DEVICE).eval()
    with torch.no_grad(), amp():
        after = reloaded(**probe).logits.float()
    diff = (before - after).abs().max().item()
    assert diff < 1e-2 * max(1.0, before.abs().max().item()), f"saved checkpoint differs (max {diff})"
    print(f"saved to {out} ({'bf16' if SAVE_BF16 else 'fp32'}); reload check passed (max logit difference {diff:.1e})")
    del reloaded
    return model


def remove_progress_files(name):
    for suffix in ["in_progress", "best"]:
        if os.path.exists(progress_path(name, suffix)):
            os.remove(progress_path(name, suffix))


@torch.no_grad()
def blend(model, other_path, alpha):
    """model <- (1 - alpha) * model + alpha * other (another checkpoint of the same architecture), weight by weight."""
    other = type(base_of(model)).from_pretrained(other_path, dtype=torch.float32)
    theirs = dict(other.named_parameters())
    for name, p in base_of(model).named_parameters():
        p.mul_(1 - alpha).add_(theirs[name].to(p.device, p.dtype), alpha=alpha)
    del other, theirs
    free_memory()
    return model


def checkpoint_path(name):
    return os.path.join(BASE_DIR, f"{RUN_NAME}_{name}")


def trained(path):
    return os.path.exists(os.path.join(path, "config.json")) or os.path.exists(os.path.join(path, "adapter_config.json"))


def examples_for(full, short=None):
    """Examples for a run of the current RUN_SIZE."""
    if QUICK_TEST:
        return int(os.environ.get("ARVL_EXAMPLES", "8"))
    return {"pilot": max(64, full // 500), "short": short or max(1_000, full // 10), "full": full}[RUN_SIZE]


def record_run(name, **info):
    _, results = results_file()
    results.setdefault("runs", {})[name] = {**info, "date": time.strftime("%Y-%m-%d %H:%M")}
    save_results(runs=results["runs"])


def run_experiment(name, recipe, base, method="lora", examples=10_000, micro_batch=4, accum=2, lr=2e-4, lrs=None, decoder=True,
                   check_names=("drawn Arabic", "drawn English"), select_on=(), patience=None, snapshot_at=(), score=True,
                   score_sets=None, label=None, note="", seed=0):
    """Train one experiment from `base`, save it as v6_<name> on Drive, score it on the suite and record it."""
    out, label = checkpoint_path(name), label or name
    adapter = method == "adapter"
    if trained(out):
        print(f"{name}: already trained ({out})")
    else:
        assert trained(base) or not os.path.isabs(base), f"{name}: its base {base} is not trained yet; run the step before it"
        if hasattr(recipe, "describe"):
            print(recipe.describe())
        torch.manual_seed(seed)
        free_memory()
        if DEVICE.type == "cuda":
            torch.cuda.reset_peak_memory_stats()
        model, processor = load_model(base)
        model = make_trainable(model, method, decoder)
        run = train(model, processor, recipe, name, examples, method, micro_batch, accum, lr, lrs, check_names, select_on, patience, snapshot_at)
        model = save_checkpoint(model, processor, out, adapter=adapter)
        record_run(name, base=base, method=method, lr=lr, lrs=lrs if method == "full" else None, note=note, seed=seed,
                   recipe=recipe.shares() if hasattr(recipe, "shares") else str(recipe), **run)
        remove_progress_files(name)
        plot_history(run["history"], name, os.path.join(BASE_DIR, f"{RUN_NAME}_{name}_curves.png"))
        del model
        free_memory()
    if score:
        if adapter:
            score_suite(base, label, sets=score_sets, adapter=out, note=note)
        else:
            score_suite(out, label, sets=score_sets, note=note)
    return out


def memory_pilot(name, recipe, base, method="full", micro_batch=4, accum=2, steps=200, lr=2e-4, lrs=None):
    """A short run that only measures peak GPU memory and speed. Nothing is saved but the measurement."""
    key = f"{name} ({method}, micro-batch {micro_batch}, freeze {FREEZE_VISION_BLOCKS})"
    _, results = results_file()
    if key in results.get("pilots", {}):
        print(f"pilot already measured: {results['pilots'][key]}")
        return results["pilots"][key]
    free_memory()
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    model = None
    try:
        model, processor = load_model(base)
        model = make_trainable(model, method)
        run = train(model, processor, recipe, f"pilot_{name}", steps * micro_batch * accum, method, micro_batch, accum, lr, lrs,
                    max_steps=steps if not QUICK_TEST else 2, checks=1, keep_files=False)
        outcome = {"fits": True, "peak_gb": round(run["peak_gpu_gb"], 1), "seconds_per_step": round(run["minutes"] * 60 / max(1, run["steps"]), 2),
                   "examples_per_hour": int(run["examples"] / max(1e-6, run["minutes"] / 60))}
    except torch.cuda.OutOfMemoryError as e:
        outcome = {"fits": False, "error": str(e)[:200]}
    finally:
        del model
        free_memory()
    results = results_file()[1]
    results.setdefault("pilots", {})[key] = outcome
    save_results(pilots=results["pilots"])
    print(f"pilot {key}: {outcome}")
    if not outcome["fits"]:
        print("Out of memory. In order: set FREEZE_VISION_BLOCKS = 8 in section A and run sections A-H again; or halve "
              "micro_batch and double accum; or lower MAX_TOKENS to 768.")
    return outcome

---
# Phase 0: score the baselines on the frozen suite

Four models, same prompts, same token limits, same scoring: **v5 stage 3** (the pure-OCR checkpoint), **v5 stage 4**
(forms), **Qwen2.5-VL-3B** as it is, and **Qwen3-VL-2B** zero-shot, the licence-clean candidate for a new base.
Expect about an hour per model on an L4 (the suite has about 9,000 line images, 400 pages and 650 forms at full
size, plus the swapped-image test). A model that fails to load (for example Qwen3-VL on an older `transformers`) is reported and skipped.

Read the table by question. The headline column is what the roadmap's gates use; the v5 numbers in the roadmap
were measured on the same KITAB images (KITAB KHATT 0.49 for stage 4 and 0.40 for stage 3 on the 112 clean lines),
so this run should reproduce them to within the interval.

In [ ]:
PHASE0 = [("v5 stage 3", V5_STAGE3), ("v5 stage 4", V5_STAGE4), ("Qwen2.5-VL-3B", BASELINE_ID)] + ([("Qwen3-VL-2B", QWEN3_VL_ID)] if SCORE_QWEN3 else [])
for label, path in PHASE0:
    try:
        score_suite(path, label, note="phase 0 baseline")
    except Exception as e:
        print(f"{label}: could not be scored ({type(e).__name__}: {str(e)[:300]})")
        free_memory()
BASELINES = [label for label, _ in PHASE0 if scores_of(label)]
suite_table(BASELINES)
form_table(BASELINES)

### Label audit (optional, about 1–2 hours)
A small CER difference means nothing if the labels are wrong. Following CER-HV, every headline line is read by two
models; a line where **both** miss its label by more than 25% CER is flagged as a likely label error. The readers
are v5 stage 3 and, for Arabic, **AIN-7B** (MIT licence, 0.03 CER on KITAB's OnlineKhatt); for English, Qwen2.5-VL-3B.
Their predictions from phase 0 are reused, so only AIN-7B runs (about 16 GB of GPU memory in bf16).

The flagged lines are saved with a review sheet per set (`v6_label_audit/<set>.png` on Drive: image, label and both
readings). You do not need to read Arabic to use it: a label that is plainly a different length from the line, or
a line cut off in the image, is visible. Every table can then show the **audited** score, without the flagged lines,
next to the plain one.

In [ ]:
RUN_LABEL_AUDIT = False
AUDIT_THRESHOLD = 0.25
ARABIC_LINE_SETS = [e["name"] for e in SUITE_ENTRIES if e["kind"] == "line" and not e.get("generated")
                    and e["question"] in ("unseen Arabic handwriting", "historical handwriting", "printed Arabic")]
ENGLISH_LINE_SETS = [e["name"] for e in SUITE_ENTRIES if e["kind"] == "line" and not e.get("generated") and e["question"] == "English"]


def saved_predictions(label, name):
    path = predictions_path(label)
    if not os.path.exists(path):
        return None
    return json.load(open(path)).get(name)


def per_line(name, predictions, truths):
    f = NORMALISERS.get(ENTRY[name].get("normaliser")) if ENTRY[name].get("normaliser") == "letters_only" else (lambda x: x)
    return [(edit_distance(f(normalise(p)), f(normalise(t))), len(f(normalise(t)))) for p, t in zip(predictions, truths)]


def set_truths(name):
    entry = ENTRY[name]
    if entry.get("generated"):
        return [a for _, _, a in entry_examples(entry)[0]]
    texts = dataset(entry["dataset"])["text"]
    rows = entry["indices"][:SUITE_N] if SUITE_N else entry["indices"]
    return [texts[r] for r in rows]


if RUN_LABEL_AUDIT:
    score_suite(AIN_ID, "AIN-7B", sets=ARABIC_LINE_SETS, visual=False, note="second reader for the label audit")
    audit, sheet_dir = {}, os.path.join(BASE_DIR, f"{RUN_NAME}_label_audit")
    os.makedirs(sheet_dir, exist_ok=True)
    for name in ARABIC_LINE_SETS + ENGLISH_LINE_SETS:
        second = "AIN-7B" if name in ARABIC_LINE_SETS else "Qwen2.5-VL-3B"
        a, b = saved_predictions("v5 stage 3", name), saved_predictions(second, name)
        if a is None or b is None:
            continue
        truths, headline = set_truths(name), set(ENTRY[name]["headline"])
        ea, eb = per_line(name, a["predictions"], truths), per_line(name, b["predictions"], truths)
        flagged = [k for k, r in enumerate(a["rows"]) if r in headline and ea[k][0] > AUDIT_THRESHOLD * max(1, ea[k][1]) and eb[k][0] > AUDIT_THRESHOLD * max(1, eb[k][1])]
        audit[name] = [a["rows"][k] for k in flagged]
        print(f"   {name}: {len(flagged)} of {len(headline)} headline lines flagged")
        if flagged:                                  # a review sheet: image, label, both readings
            examples = entry_examples(ENTRY[name])[0]
            font = get_font(ARABIC_BODY_FONTS[0], 18)
            tiles = []
            for k in flagged[:30]:
                img = examples[k][0]
                img = img.resize((min(900, img.width), max(1, int(img.height * min(900, img.width) / img.width))))
                tile = Image.new("RGB", (920, img.height + 90), "white")
                tile.paste(img, (10, 5))
                d = ImageDraw.Draw(tile)
                for j, (title, text) in enumerate([("label", truths[k]), ("v5", a["predictions"][k]), (second, b["predictions"][k])]):
                    draw_text(d, 10, 910, img.height + 8 + 27 * j, f"{text[:90]}  :{title}", font, (0, 0, 0) if j == 0 else (150, 30, 30), "right")
                tiles.append(tile)
            sheet = Image.new("RGB", (920, sum(t.height for t in tiles)), "white")
            y = 0
            for t in tiles:
                sheet.paste(t, (0, y))
                y += t.height
            sheet.save(os.path.join(sheet_dir, re.sub(r"[^\w\-]+", "_", name) + ".png"))
    save_results(label_audit={"threshold": AUDIT_THRESHOLD, "readers": "v5 stage 3 + AIN-7B (Arabic) / Qwen2.5-VL-3B (English)", "flagged": audit})
    print(f"review sheets in {sheet_dir}")


def audited_table(labels):
    """Headline CER without the lines the label audit flagged, next to the plain headline CER."""
    flagged = results_file()[1].get("label_audit", {}).get("flagged", {})
    if not flagged:
        print("no label audit yet")
        return
    print(f"{'set':36s}{'flagged':>8s}" + "".join(f"{l[:20]:>24s}" for l in labels))
    for name, rows in flagged.items():
        truths, keep = set_truths(name), set(ENTRY[name]["headline"]) - set(rows)
        cells = []
        for label in labels:
            p = saved_predictions(label, name)
            if p is None:
                cells.append("")
                continue
            pairs = [e for e, r in zip(per_line(name, p["predictions"], truths), p["rows"]) if r in keep]
            plain_cer = main_of(scores_of(label)[name])
            cells.append(f"{sum(e for e, _ in pairs) / max(1, sum(n for _, n in pairs)):.3f} ({plain_cer:.3f})")
        print(f"{name[:35]:36s}{len(rows):8d}" + "".join(f"{c:>24s}" for c in cells))
    print("cells: audited CER (plain headline CER)")

---
# Phase 1: quick wins without retraining

Four inference-time changes, each scored on the suite against the phase-0 numbers of the same checkpoint, with a
**paired** bootstrap (same images, so the interval is about the change, not about which images were drawn):

1. **Repetition control for text lines**: `no_repeat_ngram_size=4` (in v5's 100-image test it took KHATT from 0.308 to
   0.167). Forms keep plain greedy decoding, because JSON repeats short sequences.
2. **Stage 3 / stage 4 weight merge**: weights `(1 − α) · stage 3 + α · stage 4` for α = 0.3, 0.5, 0.7. The best α
   recovers stage 3's handwriting while forms stay near 0.99; it is saved as `v6_p1_merge`.
3. **Schema-forced JSON** for forms (section F): strict JSON should go to 100% at no cost in field accuracy.
4. **CLAHE** on the historical sets.

The decisions are written to `results_v6.json` as `defaults` and used by every later score. To compare later runs
fairly, the phase-0 baselines are re-scored with the chosen defaults at the end of this phase.

In [ ]:
LINE_SETS = [e["name"] for e in SUITE_ENTRIES if e["kind"] in ("line", "page")]
FORM_SETS = [e["name"] for e in SUITE_ENTRIES if e["kind"] == "form"]
HANDWRITING_SETS = [e["name"] for e in SUITE_ENTRIES if e["question"] in ("unseen Arabic handwriting", "historical handwriting")]
HISTORICAL_SETS = [e["name"] for e in SUITE_ENTRIES if e["question"] == "historical handwriting"]


def paired_cer(label_a, label_b, name, rounds=1000, seed=0):
    """CER(b) - CER(a) on the headline lines of one set, with a 95% paired bootstrap interval, from saved predictions."""
    a, b = saved_predictions(label_a, name), saved_predictions(label_b, name)
    if a is None or b is None or a["rows"] != b["rows"]:
        return None
    truths, headline = set_truths(name), set(ENTRY[name]["headline"])
    keep = [k for k, r in enumerate(a["rows"]) if r in headline]
    if len(keep) < 2:
        return None
    ea = np.array([per_line(name, [a["predictions"][k]], [truths[k]])[0] for k in keep], dtype=float)
    eb = np.array([per_line(name, [b["predictions"][k]], [truths[k]])[0] for k in keep], dtype=float)
    idx = np.random.default_rng(seed).integers(0, len(keep), size=(rounds, len(keep)))
    boot = eb[idx, 0].sum(1) / np.maximum(1, eb[idx, 1].sum(1)) - ea[idx, 0].sum(1) / np.maximum(1, ea[idx, 1].sum(1))
    diff = eb[:, 0].sum() / max(1, eb[:, 1].sum()) - ea[:, 0].sum() / max(1, ea[:, 1].sum())
    return float(diff), [float(np.percentile(boot, 2.5)), float(np.percentile(boot, 97.5))]


def compare(label_a, label_b, sets):
    """Paired CER change from a to b per set. Returns (mean change, sets clearly better, sets clearly worse)."""
    changes, better, worse = [], [], []
    print(f"{label_b}  vs  {label_a}  (negative = better)")
    for name in sets:
        r = paired_cer(label_a, label_b, name)
        if r is None:
            continue
        diff, (lo, hi) = r
        changes.append(diff)
        verdict = "better" if hi < 0 else "worse" if lo > 0 else ""
        (better if hi < 0 else worse if lo > 0 else []).append(name)
        print(f"   {name[:36]:38s}{diff:+.3f} [{lo:+.3f}, {hi:+.3f}] {verdict}")
    return (float(np.mean(changes)) if changes else float("nan")), better, worse

In [ ]:
# 1. repetition control on text lines
for label, path in [("v5 stage 3", V5_STAGE3), ("v5 stage 4", V5_STAGE4)]:
    score_suite(path, f"{label} | no-repeat-4", sets=LINE_SETS, decoding={"no_repeat_ngram_size": 4}, visual=False, note="phase 1: decoding")
decision = {}
for label in ["v5 stage 3", "v5 stage 4"]:
    mean_change, better, worse = compare(label, f"{label} | no-repeat-4", LINE_SETS)
    decision[label] = {"mean_change": mean_change, "better": better, "worse": worse}
use_no_repeat = all(d["mean_change"] < 0 and not d["worse"] for d in decision.values())
print(f"\nrepetition control for text lines: {'ADOPT' if use_no_repeat else 'do not adopt'} "
      f"(adopted when the mean change is negative for both checkpoints and no set is clearly worse)")

In [ ]:
# 2. stage 3 / stage 4 merge
MERGE_ALPHAS = [0.3, 0.5, 0.7]
line_decoding = {"no_repeat_ngram_size": 4} if use_no_repeat else {}
for alpha in MERGE_ALPHAS:
    label = f"v5 merge a{alpha}"
    if all(n in scores_of(label) for n in LINE_SETS + FORM_SETS):
        continue
    model, processor = load_model(V5_STAGE3)
    blend(model, V5_STAGE4, alpha)
    score_suite(None, label, model=model, processor=processor, decoding=line_decoding, schema=False, visual=False,
                note=f"phase 1: (1 - {alpha}) x stage 3 + {alpha} x stage 4")
    del model
    free_memory()
reference = "v5 stage 3 | no-repeat-4" if use_no_repeat else "v5 stage 3"
merge_rows = []
for alpha in MERGE_ALPHAS:
    label = f"v5 merge a{alpha}"
    mean_change, _, worse = compare(reference, label, HANDWRITING_SETS)
    forms = scores_of(label).get("Generated forms (v5 generator)", {}).get("all", {}).get("field_exact", 0)
    merge_rows.append((alpha, mean_change, forms, worse))
    print(f"   alpha {alpha}: handwriting CER change vs stage 3 {mean_change:+.3f}, v5 forms field accuracy {forms:.3f}\n")
eligible = [r for r in merge_rows if r[2] >= 0.98]
best_merge = min(eligible, key=lambda r: r[1]) if eligible else None
print(f"best merge: {'alpha ' + str(best_merge[0]) if best_merge else 'none keeps v5 forms at 0.98 or more'}")
MERGE_PATH = checkpoint_path("p1_merge")
if best_merge and not trained(MERGE_PATH):
    model, processor = load_model(V5_STAGE3)
    blend(model, V5_STAGE4, best_merge[0])
    save_checkpoint(model, processor, MERGE_PATH)
    del model
    free_memory()

In [ ]:
# 3. schema-forced JSON for forms, and 4. CLAHE on the historical sets
form_models = [("v5 stage 4", V5_STAGE4)] + ([("v5 merge (best)", MERGE_PATH)] if trained(MERGE_PATH) else [])
for label, path in form_models:
    if label == "v5 merge (best)":
        score_suite(path, label, decoding=line_decoding, schema=False, visual=False, note="phase 1: the chosen merge")
    score_suite(path, f"{label} | schema JSON", sets=FORM_SETS, schema=True, visual=False, note="phase 1: schema-forced JSON")
for label, _ in form_models:
    for name in FORM_SETS:
        a, b = scores_of(label).get(name), scores_of(f"{label} | schema JSON").get(name)
        if a and b:
            print(f"   {label:18s} {name:30s} strict JSON {a['all']['strict_json']:.2f} -> {b['all']['strict_json']:.2f}, "
                  f"fields {a['all']['field_exact']:.3f} -> {b['all']['field_exact']:.3f}")
s4 = [(scores_of("v5 stage 4").get(n), scores_of("v5 stage 4 | schema JSON").get(n)) for n in FORM_SETS]
s4 = [(a, b) for a, b in s4 if a and b]
use_schema = bool(s4) and all(b["all"]["field_exact"] >= a["all"]["field_exact"] - 0.01 for a, b in s4)
print(f"schema-forced JSON: {'ADOPT' if use_schema else 'do not adopt'} (adopted when no form set loses more than 0.01 field accuracy)")

for label, path in [("v5 stage 3", V5_STAGE3), ("v5 stage 4", V5_STAGE4)]:
    score_suite(path, f"{label} | CLAHE", sets=HISTORICAL_SETS, decoding=line_decoding, clahe_sets=HISTORICAL_SETS, visual=False, note="phase 1: CLAHE")
clahe_sets = []
for name in HISTORICAL_SETS:
    changes = [paired_cer(f"{l}{' | no-repeat-4' if use_no_repeat else ''}", f"{l} | CLAHE", name) for l in ["v5 stage 3", "v5 stage 4"]]
    changes = [c for c in changes if c]
    if changes and all(c[1][1] < 0 for c in changes):                # clearly better for both checkpoints
        clahe_sets.append(name)
    print(f"   CLAHE on {name}: " + ", ".join(f"{c[0]:+.3f} [{c[1][0]:+.3f}, {c[1][1]:+.3f}]" for c in changes))
print(f"CLAHE: adopted for {clahe_sets or 'no set'} (adopted per set when it is clearly better for both checkpoints)")

In [ ]:
# Gate 1: save the decisions, then re-score the baselines with them
DEFAULTS.update({"line_decoding": line_decoding, "form_schema": use_schema, "clahe_sets": clahe_sets,
                 "phase1_merge": MERGE_PATH if trained(MERGE_PATH) else None})
save_results(defaults=DEFAULTS)
print("defaults saved:", DEFAULTS)
for label, path in [("v5 stage 3", V5_STAGE3), ("v5 stage 4", V5_STAGE4)]:
    score_suite(path, f"{label} (phase 1 defaults)", note="baseline with the phase 1 defaults")
P1_LINE_MODEL = "v5 merge (best)" if trained(MERGE_PATH) and compare("v5 stage 3 (phase 1 defaults)", "v5 merge (best)", HANDWRITING_SETS)[0] < 0 else "v5 stage 3 (phase 1 defaults)"
P2_BASE = MERGE_PATH if P1_LINE_MODEL.startswith("v5 merge") else V5_STAGE3
print(f"phase 2 starts from: {P1_LINE_MODEL} ({P2_BASE})")
suite_table(["v5 stage 3 (phase 1 defaults)", "v5 stage 4 (phase 1 defaults)"] + (["v5 merge (best)"] if trained(MERGE_PATH) else []))

---
# Phase 2: unseen Arabic handwriting

**Recipes used from here on.** `READING_MIX` is v5's stage-3 mix with v6's sources (it keeps printed Arabic, English,
bilingual lines and scene words from regressing); `REAL_HANDWRITING` combines every real handwriting source with the
augmentation of section C. KHATT comes from the official release if you added it, else from the capped Teklia copy.

In [ ]:
def drawn(language, source=None, max_lines=1):
    return lambda rng: text_line_example(rng, language, rng.choice([0, 1, 1, 2]), max_lines=max_lines, source=source)


READING_MIX = Mixture("reading mix (v5 stage 3 shares, v6 sources)", [
    (0.24, "Arabic lines (Wikipedia)", drawn("ar", "msa")), (0.12, "diacritised lines", drawn("ar", "tashkeel")),
    (0.07, "classical lines without diacritics", drawn("ar", "classical")),
    (0.06, "blocks of 2-5 lines", lambda rng: text_line_example(rng, "ar", rng.choice([0, 1]), max_lines=5, source=rng.choice(["msa", "classical"]))),
    (0.09, f"KHATT ({KHATT_TRAIN})", line_source(KHATT_TRAIN)), (0.09, "Muharaf", line_source("muharaf")),
    (0.04, "scene words", scene_word_example), (0.05, "bilingual lines", bilingual_line_example),
    (0.10, "English lines", drawn("en")), (0.10, "IAM", line_source("iam")), (0.04, "SynthDoG pages", page_source("synthdog"))])
def real_handwriting_mix():
    return Mixture("real handwriting", [
        (0.30, f"KHATT ({KHATT_TRAIN})", line_source(KHATT_TRAIN)), (0.18, "memoir lines", line_source("memoir")),
        (0.18, "Muharaf", line_source("muharaf")), (0.12, "self-labelled lines", line_source("self_labelled")),
        (0.12, "IAM", line_source("iam")), (0.10, "GNHK", line_source("gnhk"))])


REAL_HANDWRITING = real_handwriting_mix()
CHECK_P2 = ("drawn Arabic", "drawn English", "handwriting style", "KHATT val", "memoir val", "Muharaf val", "IAM val")
print(READING_MIX.describe())
print(REAL_HANDWRITING.describe())

### 2.0 Does writer diversity help? (needs the official KHATT release, about 8 L4 hours)
The stage-3 checkpoint is fine-tuned on 1,400 KHATT lines chosen three ways, two seeds each, and scored on the
official unique-text test (unseen writers, unseen text):

| Arm | Lines | Varies |
| --- | --- | --- |
| A | all unique-text lines of about 150 writers | text, few writers |
| B | 2 unique-text lines from each of 700 writers | text and writers |
| C | 2 fixed-passage lines from each of 700 writers | writers, same text |

B beating A says more writers help; B beating C says more texts help. If B wins, phase 2's budget goes to sources
that add writers. If nothing wins, more of the same data will not close the gap, and the method changes come first.

In [ ]:
def writer_arms(ds, n_lines=1400, seed=0):
    writers, kinds = list(ds["writer"]), list(ds["note"])
    unique, fixed = defaultdict(list), defaultdict(list)
    for i, (w, k) in enumerate(zip(writers, kinds)):
        (fixed if k == "fixed" else unique)[w].append(i)
    rng = random.Random(seed)
    pool = sorted(unique)
    rng.shuffle(pool)
    arm_a = []
    for w in pool:
        if len(arm_a) >= n_lines:
            break
        arm_a += unique[w]
    two = lambda groups: [i for w in pool for i in rng.sample(groups[w], min(2, len(groups[w])))][:n_lines]
    return {"A": arm_a[:n_lines], "B": two(unique), "C": two(fixed)}


def subset_source(ds, rows):
    def example(rng):
        row = ds[rng.choice(rows)]
        img = row["image"].convert("RGB")
        if rng.random() < 0.8:
            img = augment_line(img, rng)
        return damage(img, rng, rng.choice([0, 0, 1])), OCR_INSTRUCTION, row["text"]
    return example


WRITER_SETS = ["KHATT unique-text test (official)", "KHATT seen-writer test (official)", "KITAB KHATT", "KITAB OnlineKhatt"]
official = TRAIN["khatt_official"]
if official is None or not any(official["writer"]):
    print("skipped: needs the official KHATT release with writer IDs (part 1b, section 3b)")
else:
    arms = writer_arms(official)
    for arm, rows in arms.items():
        print(f"arm {arm}: {len(rows)} lines from {len({official['writer'][i] for i in rows})} writers")
    for arm, rows in arms.items():
        for seed in (0, 1):
            run_experiment(f"p2_writers_{arm}{seed}", subset_source(official, rows), base=V5_STAGE3, method="lora", examples=examples_for(4_200, 2_000),
                           lr=2e-4, check_names=("KHATT val",), score_sets=WRITER_SETS, seed=seed, note=f"writer-diversity arm {arm}, seed {seed}")
    name = "KHATT unique-text test (official)"
    for arm in "ABC":
        values = [main_of(scores_of(f"p2_writers_{arm}{s}")[name]) for s in (0, 1) if name in scores_of(f"p2_writers_{arm}{s}")]
        print(f"arm {arm}: unseen-writer CER {np.mean(values):.3f} (seeds: {', '.join(f'{v:.3f}' for v in values)})")
    for a, b in [("A", "B"), ("C", "B")]:
        for seed in (0, 1):
            r = paired_cer(f"p2_writers_{a}{seed}", f"p2_writers_{b}{seed}", name)
            if r:
                print(f"   B minus {a}, seed {seed}: {r[0]:+.3f} [{r[1][0]:+.3f}, {r[1][1]:+.3f}]  (negative: B is better)")

### 2.1 Self-labelling unlabelled handwriting (optional)
Lines without transcriptions (your `unlabelled_lines/` folder, and the memoir blind-test lines if they carry no
text) are read by our best model and by AIN-7B. A line is kept only when the two readings agree within 10% CER (the
AIN reading is kept as its label), and the result becomes the "self-labelled lines" source of `REAL_HANDWRITING`,
capped at 12% of it. The kept lines are saved on Drive (`v6_self_labelled`) and reused in later sessions.

In [ ]:
RUN_SELF_LABELLING = False
AGREEMENT_CER = 0.10
if RUN_SELF_LABELLING:
    from datasets import concatenate_datasets, Dataset, Features, Value, Image as ImageFeature
    pools = [d for d in [dataset("unlabelled_lines"), dataset("memoir_blind")] if d is not None and len(d) and not any(d.select(range(min(20, len(d))))["text"])]
    if not pools:
        print("nothing to self-label: add line images to MyDrive/arabic_vlm/unlabelled_lines/ and run part 1b, section 8")
    else:
        lines = concatenate_datasets([p.select_columns(["image"]) for p in pools])
        examples = [(lines[i]["image"].convert("RGB"), OCR_INSTRUCTION, "") for i in range(len(lines))]
        readings = {}
        for label, path in [("ours", P2_BASE), ("AIN-7B", AIN_ID)]:
            model, processor = load_model(path, dtype=AMP_DTYPE if DEVICE.type == "cuda" else torch.float32)
            readings[label] = read_images(model, processor, examples, decoding=DEFAULTS["line_decoding"])
            del model
            free_memory()
        keep = [i for i, (a, b) in enumerate(zip(readings["ours"], readings["AIN-7B"]))
                if b.strip() and edit_distance(normalise(a), normalise(b)) <= AGREEMENT_CER * max(1, len(normalise(b)))]
        print(f"self-labelling: {len(keep):,} of {len(lines):,} lines kept (the two readings agree within {AGREEMENT_CER:.0%} CER)")
        if keep:
            kept = lines.select(keep).add_column("text", [readings["AIN-7B"][i] for i in keep])
            kept.save_to_disk(SELF_LABELLED_DIR)
            TRAIN["self_labelled"] = load_from_disk(SELF_LABELLED_DIR)
            REAL_HANDWRITING = real_handwriting_mix()
            print(REAL_HANDWRITING.describe())

### 2.2 Pre-train on rendered handwriting-style lines (about 40 L4 hours at full size)
70% rendered handwriting-style lines (section B), 10% real handwriting, 20% the reading mix, from the phase-1 line
model, with v5's LoRA recipe. HATFormer's 1M synthetic lines were worth 4.2 CER points on manuscripts; the roadmap
starts at 300,000 and grows toward 1,000,000 if it keeps paying. A snapshot at 20% of the run is kept for the phase 6
checkpoint merge. The best check on the handwriting validation sets is kept.

In [ ]:
P2A = Mixture("phase 2.2: rendered handwriting-style lines", [
    (0.70, "rendered handwriting-style lines (80% Arabic)", handwriting_style_example),
    (0.10, "real handwriting", REAL_HANDWRITING), (0.20, "reading mix", READING_MIX)])
show_examples(P2A)
run_experiment("p2a_rendered", P2A, base=P2_BASE, method="lora", examples=examples_for(300_000, 30_000), lr=2e-4,
               check_names=CHECK_P2, select_on=("handwriting style", "KHATT val", "memoir val"), snapshot_at=(0.2,),
               note="phase 2: rendered handwriting-style pre-training")

### 2.3 Real handwriting, twice: v5's LoRA recipe and full fine-tuning
The same data, two methods, so the data gain and the method gain are credited separately: about 45,000 real lines,
three passes, each line augmented, plus 10% rendered lines and 10% reading mix. The full fine-tuning run first runs
a 200-step **memory pilot**; if it does not fit in the L4's 24 GB, set `FREEZE_VISION_BLOCKS = 8` in section A and
run sections A–H again. Each run is followed by a short mixed stage (2.4), as the roadmap's sequence asks.

In [ ]:
P2B = Mixture("phase 2.3: real handwriting", [(0.80, "real handwriting (augmented)", REAL_HANDWRITING),
                                              (0.10, "rendered handwriting-style lines", handwriting_style_example), (0.10, "reading mix", READING_MIX)])
P2B_BASE = checkpoint_path("p2a_rendered")
run_experiment("p2b_real_lora", P2B, base=P2B_BASE, method="lora", examples=examples_for(135_000, 15_000), lr=2e-4,
               check_names=CHECK_P2, select_on=("KHATT val", "memoir val", "Muharaf val"), snapshot_at=(0.2,), note="phase 2: real handwriting, LoRA")

In [ ]:
pilot = memory_pilot("p2b_real_full", P2B, base=P2B_BASE, method="full", micro_batch=4, accum=2)
if pilot["fits"]:
    run_experiment("p2b_real_full", P2B, base=P2B_BASE, method="full", examples=examples_for(135_000, 15_000), micro_batch=4, accum=2,
                   lrs=FULL_LR, check_names=CHECK_P2, select_on=("KHATT val", "memoir val", "Muharaf val"), snapshot_at=(0.2,),
                   note="phase 2: real handwriting, full fine-tuning")

### 2.4 A short mixed stage after each, and gate 2

In [ ]:
P2C = Mixture("phase 2.4: short mixed stage", [(0.35, "real handwriting", REAL_HANDWRITING), (0.65, "reading mix", READING_MIX)])
for method in ["lora", "full"]:
    if trained(checkpoint_path(f"p2b_real_{method}")):
        run_experiment(f"p2c_mixed_{method}", P2C, base=checkpoint_path(f"p2b_real_{method}"), method=method, examples=examples_for(20_000, 4_000),
                       lr=1e-4, lrs={k: v / 2 for k, v in FULL_LR.items()}, check_names=CHECK_P2, note=f"phase 2: short mixed stage ({method})")
P2_LABELS = [l for l in ["v5 stage 3 (phase 1 defaults)", "p2a_rendered", "p2b_real_lora", "p2c_mixed_lora", "p2b_real_full", "p2c_mixed_full"] if scores_of(l)]
suite_table(P2_LABELS, questions=["unseen Arabic handwriting", "historical handwriting", "printed Arabic", "English"])
if scores_of("p2c_mixed_lora") and scores_of("p2c_mixed_full"):
    print()
    compare("p2c_mixed_lora", "p2c_mixed_full", HANDWRITING_SETS)
for label in P2_LABELS[1:]:
    gates(label, baseline="v5 stage 3 (phase 1 defaults)")

---
# Phase 3: historical handwriting
The 2026 winner's order: manuscripts first, then the target styles, then a short mix, with the vision encoder tuned
at a low rate. Images are converted to grey (as the winner did), or given CLAHE if phase 1 adopted it for the
historical sets. The phase starts from the better of the two phase-2 runs on the handwriting sets (or set `P3_BASE`
yourself), with the same method.

In [ ]:
P2_FINALS = [l for l in ["p2c_mixed_full", "p2c_mixed_lora"] if scores_of(l)]
if len(P2_FINALS) == 2:
    better_full = compare("p2c_mixed_lora", "p2c_mixed_full", HANDWRITING_SETS)[0] < 0
    P3_LABEL = "p2c_mixed_full" if better_full else "p2c_mixed_lora"
else:
    P3_LABEL = P2_FINALS[0] if P2_FINALS else None
P3_BASE = checkpoint_path(P3_LABEL) if P3_LABEL else P2_BASE
P3_METHOD = "full" if P3_LABEL == "p2c_mixed_full" else "lora"
LOOK = "clahe" if DEFAULTS["clahe_sets"] else "gray"
print(f"phase 3 starts from {P3_LABEL or 'the phase-1 line model'} with {P3_METHOD}; historical images: {LOOK}")
P3A = Mixture("phase 3.1: Muharaf", [(0.80, f"Muharaf ({LOOK})", line_source("muharaf", look=LOOK)), (0.20, "reading mix", READING_MIX)])
P3B = Mixture("phase 3.2: AraMS-28k and the memoirs", [
    (0.40, f"AraMS-28k, letters-only instruction ({LOOK})", line_source("arams", OCR_LETTERS_INSTRUCTION, look=LOOK)),
    (0.40, f"memoir lines ({LOOK})", line_source("memoir", look=LOOK)), (0.20, "reading mix", READING_MIX)])
P3C = Mixture("phase 3.3: short mix", [
    (0.15, "Muharaf", line_source("muharaf", look=LOOK)), (0.10, "AraMS-28k", line_source("arams", OCR_LETTERS_INSTRUCTION, look=LOOK)),
    (0.10, "memoirs", line_source("memoir", look=LOOK)), (0.25, "modern handwriting", REAL_HANDWRITING), (0.40, "reading mix", READING_MIX)])
CHECK_P3 = ("drawn Arabic", "Muharaf val", "AraMS val", "memoir val", "KHATT val")
step_lrs = {k: v / 2 for k, v in FULL_LR.items()}
run_experiment("p3a_muharaf", P3A, base=P3_BASE, method=P3_METHOD, examples=examples_for(50_000, 6_000), lr=2e-4, lrs=FULL_LR,
               check_names=CHECK_P3, select_on=("Muharaf val",), snapshot_at=(0.2,), note="phase 3: Muharaf")
run_experiment("p3b_arams_memoirs", P3B, base=checkpoint_path("p3a_muharaf"), method=P3_METHOD, examples=examples_for(60_000, 6_000), lr=2e-4,
               lrs=FULL_LR, check_names=CHECK_P3, select_on=("AraMS val", "memoir val"), snapshot_at=(0.2,), note="phase 3: AraMS-28k and memoirs")
run_experiment("p3c_mix", P3C, base=checkpoint_path("p3b_arams_memoirs"), method=P3_METHOD, examples=examples_for(15_000, 3_000), lr=1e-4,
               lrs=step_lrs, check_names=CHECK_P3, note="phase 3: short mix")
P3_LABELS = [l for l in [P3_LABEL, "p3a_muharaf", "p3b_arams_memoirs", "p3c_mix"] if l and scores_of(l)]
suite_table(P3_LABELS, questions=["historical handwriting", "unseen Arabic handwriting", "printed Arabic"])
if scores_of("p3c_mix"):
    gates("p3c_mix", baseline="v5 stage 3 (phase 1 defaults)")

---
# Phase 4: English, printed and handwritten
English was 10% of v5's mix and mostly short lines. Here printed English rises to 25% (lines up to 8-line
paragraphs, SynthDoG pages, olmOCR pages if part 1b took them), English handwriting to 15% (IAM and GNHK), and the
other 60% keeps the Arabic reading and handwriting mix so nothing regresses. IAM is reported raw and with punctuation
spacing normalised.

In [ ]:
P4_BASE = checkpoint_path("p3c_mix") if trained(checkpoint_path("p3c_mix")) else P3_BASE
P4 = Mixture("phase 4: English", [
    (0.12, "English lines and paragraphs (1-8 lines)", lambda rng: text_line_example(rng, "en", rng.choice([0, 1, 1, 2]), max_lines=rng.choice([1, 1, 4, 8]))),
    (0.08, "SynthDoG pages", page_source("synthdog")), (0.05, "olmOCR pages", page_source("olmocr")),
    (0.08, "IAM", line_source("iam")), (0.07, "GNHK", line_source("gnhk")),
    (0.35, "reading mix", READING_MIX), (0.25, "real handwriting", REAL_HANDWRITING)])
run_experiment("p4_english", P4, base=P4_BASE, method=P3_METHOD, examples=examples_for(40_000, 6_000), lr=1e-4, lrs=step_lrs,
               check_names=("drawn English", "IAM val", "GNHK val", "SynthDoG val", "drawn Arabic", "KHATT val"),
               select_on=("drawn English", "IAM val", "GNHK val"), note="phase 4: English")
suite_table([l for l in [os.path.basename(P4_BASE).replace(f"{RUN_NAME}_", ""), "p4_english"] if scores_of(l)], questions=["English", "printed Arabic", "unseen Arabic handwriting"])
if scores_of("p4_english"):
    gates("p4_english", baseline="v5 stage 3 (phase 1 defaults)")

---
# Phase 5: forms as a separate adapter
The JSON skill goes into its own LoRA adapter on top of the best reading model, instead of a stage that changes every
weight: plain transcription keeps using the reading model untouched, and forms load the adapter (in v5, stage 4 lost
handwriting accuracy while learning JSON). Data: 60% generator v2, 25% filled training templates (if you marked
templates in part 1b), 15% v5 forms. Scored on every form set with and without schema-forced decoding.

**Two-step reading, oracle version.** For the held-out forms the value boxes are known, so each value can be cropped,
read as a line by the reading model, and normalised (dates to ISO, phone numbers to digits, Western digits) by code.
That is an upper bound for a two-step reader: if it beats end-to-end reading by a wide margin, a field or line
detector is worth building; if not, end-to-end is enough.

In [ ]:
P5_BASE = next((checkpoint_path(n) for n in ["p4_english", "p3c_mix", "p2c_mixed_full", "p2c_mixed_lora"] if trained(checkpoint_path(n))), P2_BASE)
P5 = Mixture("phase 5: forms adapter", [(0.60, "form generator v2", form_v2_example),
                                        (0.25, "filled training templates", template_form_example if TRAIN["forms_template"] is not None else None),
                                        (0.15, "v5 forms", lambda rng: form_example(rng, rng.choice([0, 1, 1, 2])))])
print(f"forms adapter on {P5_BASE}")
ADAPTER = run_experiment("p5_forms_adapter", P5, base=P5_BASE, method="adapter", examples=examples_for(20_000, 3_000), micro_batch=2, accum=4,
                         lr=1e-4, check_names=("forms v2", "template forms", "forms v5"), select_on=("forms v2", "template forms"),
                         score_sets=FORM_SETS, label="forms adapter", note="phase 5: forms adapter")
score_suite(P5_BASE, "forms adapter | schema JSON", sets=FORM_SETS, adapter=ADAPTER, schema=True, note="phase 5: adapter + schema decoding")


def normalise_field(kind, text):
    """What a two-step reader does after reading one value: Western digits, dates to YYYY-MM-DD, phone numbers to digits."""
    t = normalise(text).translate(TO_WESTERN_DIGITS)
    if not t:
        return None
    if kind in ("date", "dob", "past_date", "future_date"):
        months = {m.lower(): i + 1 for i, m in enumerate(VOCAB["months_en"])} | {m: i + 1 for i, m in enumerate(VOCAB["months_ar"])}
        for pattern, order in [(r"(\d{4})[/\-.](\d{1,2})[/\-.](\d{1,2})", "ymd"), (r"(\d{1,2})[/\-.](\d{1,2})[/\-.](\d{4})", "dmy")]:
            m = re.search(pattern, t)
            if m:
                parts = dict(zip(order, map(int, m.groups())))
                return f"{parts['y']:04d}-{parts['m']:02d}-{parts['d']:02d}"
        m = re.search(r"(\d{1,2})\s+(\S+)\s+(\d{4})", t)
        if m and m.group(2).lower() in months:
            return f"{int(m.group(3)):04d}-{months[m.group(2).lower()]:02d}-{int(m.group(1)):02d}"
        return t
    if kind in ("phone", "mobile"):
        digits = re.sub(r"\D", "", t)
        return "0" + digits[3:] if digits.startswith("971") else digits
    if kind in ("amount", "amount_small"):
        m = re.search(r"\d[\d,]*(?:\.\d+)?", t)
        return f"{float(m.group(0).replace(',', '')):.2f}" if m else t
    if kind in ("po_box", "quantity"):
        return re.sub(r"\D", "", t) or t
    return t


@torch.no_grad()
def oracle_two_step(path, label, sets=("Held-out forms (digital)", "Held-out forms (Augraphy)")):
    model, processor = load_model(path, dtype=AMP_DTYPE if DEVICE.type == "cuda" else torch.float32)
    for name in sets:
        if name not in ENTRY:
            continue
        ds = dataset(ENTRY[name]["dataset"])
        crops, where, examples, kinds = [], [], [], []
        for r in ENTRY[name]["indices"][:SUITE_N] if SUITE_N else ENTRY[name]["indices"]:
            row = ds[r]
            img, boxes, truth = row["image"].convert("RGB"), json.loads(row["boxes"]), json.loads(row["json"])
            examples.append((img, form_instruction_v6(list(truth)), row["json"]))
            kinds.append(json.loads(row["kinds"]))
            for key, (x0, y0, x1, y1) in boxes.items():
                crops.append((img.crop((max(0, x0 - 4), max(0, y0 - 4), min(img.width, x1 + 4), min(img.height, y1 + 4))), OCR_INSTRUCTION, ""))
                where.append((len(examples) - 1, key))
        readings = read_images(model, processor, crops, decoding={})
        answers = [{} for _ in examples]
        for (i, key), text in zip(where, readings):
            answers[i][key] = normalise_field(FIELD_KIND.get(re.sub(r"_\d+$", "", key), "free_text"), text)
        predictions = [json.dumps(a, ensure_ascii=False) for a in answers]
        scores = form_scores(predictions, examples, kinds)
        results = results_file()[1]
        results.setdefault("scores", {}).setdefault(label, {"config": {"path": path, "note": "oracle two-step reading"}, "sets": {}})
        results["scores"][label]["sets"][name] = {"images": len(examples), "headline_images": len(examples), "all": scores, "headline": scores}
        save_results(scores=results["scores"])
        print(f"  {label}: {name}: fields {scores['field_exact']:.3f} (printed {scores.get('field_exact_printed', float('nan')):.3f}, "
              f"handwritten {scores.get('field_exact_handwritten', float('nan')):.3f}), whole form {scores['whole_form']:.2f}")
    del model
    free_memory()


if any(n in ENTRY for n in ("Held-out forms (digital)", "Held-out forms (Augraphy)")) and not scores_of("two-step oracle"):
    oracle_two_step(P5_BASE, "two-step oracle")
FORM_LABELS = [l for l in ["v5 stage 4 (phase 1 defaults)", "v5 stage 4 | schema JSON", "forms adapter", "forms adapter | schema JSON", "two-step oracle", "Qwen2.5-VL-3B"] if scores_of(l)]
form_table(FORM_LABELS)
if scores_of("forms adapter | schema JSON"):
    gates("forms adapter | schema JSON")

---
# Phase 6: model and training upgrades

**Full fine-tuning** ran in phase 2 (2.3) against LoRA on the same data; the comparison is printed at 2.4.

**Checkpoint merging.** Every long run saved its weights at 20% of the way (`snapshot_at`). The 2026 winner gained
2 CER points by averaging an early and a late checkpoint of one run. The cell below rebuilds each snapshot, averages
it with the run's final weights (half and half), scores the result, and keeps it (`v6_<run>_merged`) if it is better
on the run's own target sets.

In [ ]:
def snapshot_merge(name, alpha=0.5, target_sets=None):
    run = results_file()[1].get("runs", {}).get(name)
    snap = progress_path(name, "snap20")
    label = f"{name} + early snapshot"
    if run is None or not os.path.exists(snap) or not trained(checkpoint_path(name)):
        print(f"{name}: no snapshot to merge")
        return
    if not scores_of(label):
        model, processor = load_model(run["base"])
        model = make_trainable(model, run["method"])
        model.load_state_dict(torch.load(snap, map_location="cpu"), strict=False)
        if hasattr(model, "merge_and_unload"):
            model = model.merge_and_unload()
        model = model.to(torch.float32)
        blend(model, checkpoint_path(name), 1 - alpha)            # alpha of the snapshot, the rest from the final weights
        score_suite(None, label, model=model, processor=processor, visual=False, note=f"{alpha} x snapshot at 20% + {1 - alpha} x final")
        if compare(name, label, target_sets or HANDWRITING_SETS)[0] < 0 and not trained(checkpoint_path(f"{name}_merged")):
            save_checkpoint(model, processor, checkpoint_path(f"{name}_merged"))
        del model
        free_memory()
    compare(name, label, target_sets or HANDWRITING_SETS)


for name, targets in [("p2a_rendered", HANDWRITING_SETS), ("p2b_real_lora", HANDWRITING_SETS), ("p2b_real_full", HANDWRITING_SETS),
                      ("p3a_muharaf", HISTORICAL_SETS), ("p3b_arams_memoirs", HISTORICAL_SETS)]:
    if trained(checkpoint_path(name)):
        snapshot_merge(name, target_sets=targets)

**A newer base as a comparison arm (experimental, off by default).** Qwen3-VL-2B-Instruct (Apache 2.0, a 300M
SigLIP-2 encoder) fine-tuned with LoRA on the phase-2 real-handwriting recipe. The loading, LoRA targets and processor
sizes in this notebook handle it, but this path has not been run on the real model yet: a 20-step pilot runs first,
and full fine-tuning of a 2B model does not fit an L4 (fp32 weights, gradients and optimizer state alone are about
21 GB). Its zero-shot score is in phase 0.

In [ ]:
RUN_QWEN3_ARM = False
if RUN_QWEN3_ARM:
    pilot = memory_pilot("p6_qwen3vl_lora", P2B, base=QWEN3_VL_ID, method="lora", micro_batch=2, accum=4, steps=20)
    if pilot["fits"]:
        run_experiment("p6_qwen3vl_lora", P2B, base=QWEN3_VL_ID, method="lora", examples=examples_for(135_000, 15_000), micro_batch=2, accum=4,
                       lr=2e-4, check_names=CHECK_P2, select_on=("KHATT val", "memoir val", "Muharaf val"), note="phase 6: Qwen3-VL-2B arm")

**Not in this notebook yet, and why.**

| Roadmap item | Status |
| --- | --- |
| Stronger Arabic decoder: continued pre-training of Qwen2.5-0.5B | the switch is in the v5 notebook (`PRETRAIN_DECODER = True`, run with `RUN_NAME = "v5b"`); use its stage-3 checkpoint as `P2_BASE` here |
| Swap the decoder for Qwen3-0.6B | needs re-assembling the model (part 1) and re-running stages 1 and 2; worth it only if real documents show decoder errors |
| CRNN from scratch (CER-HV's architecture) | a separate notebook: different model, data loader and CTC training; it would also serve as the independent second reader for the label audit |
| GRPO with an edit-distance reward | last, and only once supervised training has plateaued; needs TRL's GRPO trainer for vision models |
| Two-step page reading with a real detector | the oracle above measures the ceiling first; a detector is built only if the ceiling is far above end-to-end |
| Two-page forms | the model takes one image per question here |

---
# Results: every run on one table

In [ ]:
_, results = results_file()
ALL_LABELS = [l for l in results.get("scores", {}) if " | " not in l and not l.startswith("p2_writers")]
print("scored:", ", ".join(ALL_LABELS))
SHOW = [l for l in ["v5 stage 3 (phase 1 defaults)", "v5 stage 4 (phase 1 defaults)", "Qwen2.5-VL-3B", "Qwen3-VL-2B",
                    "p2c_mixed_lora", "p2c_mixed_full", "p3c_mix", "p4_english"] if l in ALL_LABELS]
suite_table(SHOW)
audited_table(SHOW)
form_table(FORM_LABELS if "FORM_LABELS" in dir() else SHOW)
print("\nruns:")
for name, run in results.get("runs", {}).items():
    print(f"   {name:24s} {run['method']:8s} {run['examples']:>9,} examples {run['minutes']:>7.0f} min  peak {run.get('peak_gpu_gb', 0):.1f} GB"
          f"  best check: step {run.get('best_step')}{'  (stopped early)' if run.get('stopped_early') else ''}")
for key, pilot in results.get("pilots", {}).items():
    print(f"   pilot {key}: {pilot}")

---
# Try a model on your own image
Upload an image in Colab's Files panel and set its name below. Text is read by the best reading model; forms by the
same model with the forms adapter and schema-forced decoding, for the keys you list.

In [ ]:
MY_IMAGE = "my_form.png"
MY_KEYS = ["full_name", "date", "phone", "amount_aed"]
if os.path.exists(MY_IMAGE):
    reader = P5_BASE if "P5_BASE" in dir() else V5_STAGE3
    model, processor = load_model(reader, dtype=AMP_DTYPE if DEVICE.type == "cuda" else torch.float32, max_tokens=MAX_TOKENS_INFER)
    img = ImageOps.exif_transpose(Image.open(MY_IMAGE)).convert("RGB")
    print("OCR:", read_images(model, processor, [(img, OCR_INSTRUCTION, "")], kind="page")[0])
    adapter = checkpoint_path("p5_forms_adapter")
    if trained(adapter):
        model = PeftModel.from_pretrained(model, adapter)
        example = (img, form_instruction_v6(MY_KEYS), json.dumps({k: None for k in MY_KEYS}))
        print("JSON:", read_forms_schema(model, processor, [example])[0])
    del model
    free_memory()
else:
    print(f"Upload an image named {MY_IMAGE} to try the model (or change MY_IMAGE).")

## Notes

- **Session start.** Run sections A–H (about 10–15 minutes), then the phase you are on. Finished experiments are
  skipped; an interrupted one resumes from its last check (`v6_<name>_in_progress.pt` on Drive).
- **Starting an experiment again from scratch.** Delete `v6_<name>` (the checkpoint folder) and its
  `v6_<name>_in_progress.pt`/`_best.pt` files on Drive. To score a label again, pass `rescore=True` to `score_suite`.
- **Out of memory.** Halve `micro_batch` and double `accum` in that `run_experiment` call (same examples per step);
  for full fine-tuning set `FREEZE_VISION_BLOCKS = 8`; or lower `MAX_TOKENS` to 768 (v4 used 640).
- **Drive space.** Each checkpoint is about 2.3 GB in bf16; snapshot and best-check files are kept next to them
  (about 0.5 GB for LoRA runs, 4.6 GB for full fine-tuning). Delete the ones you no longer need once a phase is decided.
- **Licences.** The vision encoder is under the Qwen Research License, and KHATT, Muharaf, AraMS-28k and IAM are for
  research only, so a model trained here is for research use. The memoir lines and GNHK are CC BY 4.0 (attribution).